<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L10-agent-identity/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L10-agent-identity/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T07-L10-agent-identity/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T07-L10-agent-identity/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T07-L10 · Give your agent its own identity: from a static key to a delegated, scoped, expiring credential

**You will build:** the checks that turn a bearer token into an authorisation decision, the exchange that mints
an agent's token on behalf of a user, the resource-server pipeline with revocation and replay protection, and
the runner that fires a fixed list of attacks at two designs and counts what each one stops.

**Time:** ~60 minutes · **Runs on:** a laptop CPU, standard library only · **Prerequisites:** T07-L02-mcp-from-the-wire-up, T07-L08-mcp-vetting

A common starting point is one long-lived API key in an environment variable: the agent *is* the user, to every
service, for ever. This lesson replaces it with a short-lived token that names the user, names the agent acting
for them, carries only the scopes the job needs, and is good for one service. Tokens are signed with HMAC from the
standard library, so no server is needed; **this is a teaching model of the checks, not a security product**
(section 9 says what a real deployment adds).

By the end you will be able to:

1. Implement a token verifier that checks algorithm, signature, expiry, audience and scope, in a stated order, and names the reason it refused.
2. Implement an on-behalf-of exchange that issues an agent a narrower token carrying an `act` claim, and refuses escalation.
3. Implement a resource-server pipeline with revocation, single-use replay protection and an audit record of who acted for whom.
4. Measure, with exact counts, which attacks a static key and a delegated token each stop, and which neither stops.
5. Explain why a short lifetime bounds a stolen token without preventing its use inside that lifetime.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T07-L02`, `T07-L08`

**This lesson assumes you can already:**

- write a Python function that raises a custom exception carrying a short code, and catch it by that code
- decode base64 text and parse JSON in Python, and compute an HMAC with the hmac and hashlib modules
- say what a bearer token or an API key is, and what it means for a service to accept one

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [OAuth 2.0 Token Exchange (RFC 8693)](https://www.rfc-editor.org/rfc/rfc8693.html) · IETF · *free* — the standard behind exercise 2; section 4.1 defines the act claim that names the agent acting for a user
- [JSON Web Token Best Current Practices (RFC 8725)](https://www.rfc-editor.org/rfc/rfc8725.html) · IETF · *free* — why exercise 1 lets the verifier, not the token's header, choose the algorithm (section 3.1)
- [MCP specification: Authorization](https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization) · Model Context Protocol · *free* — what the revision current when this lesson was written requires of a server about token audience
- [hmac: keyed-hashing for message authentication](https://docs.python.org/3/library/hmac.html) · Python documentation · *free* — compare_digest, the constant-time comparison exercise 1 needs, and what it accepts

In [ ]:
# Setup: standard library only; versions printed.
import base64
import hashlib
import hmac
import json
import sys
from collections import namedtuple
from typing import Callable

print("python", sys.version.split()[0])

_FAILED_CHECKS: list = []
_STATUS: dict = {}          # label -> "passed" | "failed" | "not started", for the progress board
_WAITING_ON: dict = {}      # label -> the stub whose NotImplementedError stopped it

# The exercises, in the order the notebook meets them: (label, name shown, stubs, where).
_EXERCISES = [("exercise 1", "verify_token", ("verify_token",), "section 3"),
              ("exercise 2", "exchange", ("exchange",), "section 4"),
              ("exercise 3", "ResourceServer.handle", ("handle",), "section 5"),
              ("exercise 4", "run_suite and compare", ("run_suite", "compare"), "section 7")]
_SHOWN = {label: shown for label, shown, _, _ in _EXERCISES}
_STUBS = {label: stubs for label, _, stubs, _ in _EXERCISES}


def _unfinished(exc: BaseException) -> str:
    """The name of the function that raised NotImplementedError: the innermost frame."""
    tb = exc.__traceback__
    while tb is not None and tb.tb_next is not None:
        tb = tb.tb_next
    return tb.tb_frame.f_code.co_name if tb is not None else ""


def _named(labels: list) -> str:
    """["exercise 3"] -> "exercise 3 (run_episode)"; several -> "exercises 3 and 4"; a demo -> its own name."""
    if len(labels) > 1 and all(label in _SHOWN for label in labels):
        nums = [label.split()[-1] for label in labels]
        return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]
    parts = [f"{label} ({_SHOWN[label]})" if label in _SHOWN else label for label in labels]
    return parts[0] if len(parts) == 1 else ", ".join(parts[:-1]) + " and " + parts[-1]


# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: it compares the process's resident memory with what it was when the
#    block started, and stops the block as soon as the growth passes `memory_mib`. After a memory
#    stop it looks every 5 ms and stops again at once only if the memory is still growing (a bare
#    `except:` swallowed the stop; clean-up that itself allocates over 32 MiB looks the same);
#    otherwise the stop repeats after 0.25 s, as a time stop does, so `finally:` blocks and
#    context managers can clean up. It looks after 0.01 s,
#    then every 0.02 s, and more often (down to every 2 ms) once the growth passes a quarter of
#    the limit or is fast enough to get there soon: numpy can add a gigabyte in a tenth of a
#    second. A loop that appends to a list reaches gigabytes in seconds, long before
#    any sensible time limit, so the time limit alone does not bound it. The limit is 768 MiB:
#    mybinder.org gives a whole notebook 2 GB, the heaviest lesson holds about 800 MiB before a
#    check starts, and a runaway check runs twice in "Run all" (its cell, then the board). The
#    largest growth a correct answer needs in any lesson is 512 MiB. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and (not state["fired"] or state.get("memory")):
            due.append(now + state.get("poll", 0.01))    # and look at the memory: first after 0.01 s, then as _fire says
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and (not state["fired"] or state.get("memory")) else 0
        if used_mib and not state["fired"]:              # close to the limit, or fast enough to come close soon
            rate = (grown - state.get("grown", 0)) / max(now - state.get("seen", started), 0.001)   # MiB a second
            state["grown"], state["seen"] = grown, now
            state["poll"] = (0.02 if grown <= memory_mib / 4 and rate * 0.02 <= (memory_mib - grown) / 4 else   # near: by
                                         max(0.002, min(0.01, (memory_mib - grown) / max(rate, 1.0) / 4)))  # a 1/4 of the time left
        if grown > memory_mib and state["next"] is not None and not state["fired"]:   # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
            state["memory"], state["at_stop"], state["poll"] = True, grown, 0.005   # then look every 5 ms
        elif state.get("memory") and state["fired"] and grown - state["at_stop"] > 32 and state["next"] is not None:
            state["next"], state["at_stop"] = now, grown        # still growing: a bare `except:` swallowed the stop
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 10     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 768      # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet says so, by name. A wrong answer prints the check's own message,
    which names the likely mistake, and the notebook carries on, so one broken exercise never hides the
    feedback on the others. A demo names the exercises it `needs`: until each has passed its check, the
    demo says which one it is waiting for and skips. Every outcome is recorded for the progress board, and
    a script run exits non-zero at the end if any check came back wrong. A check that runs longer than
    its limit is stopped and counts as failed, so a loop that never ends cannot hang the notebook.
    """
    waiting = [name for name in needs if _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped, needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        stub = _unfinished(exc)
        _STATUS[label], _WAITING_ON[label] = "not started", stub
        owner = [name for name, stubs in _STUBS.items() if stub in stubs and name != label]
        if owner:
            print(f"{label}: skipped, needs {_named(owner)} first; it still raises NotImplementedError.")
        elif label in _SHOWN:
            print(f"{label}: not implemented yet, `{stub or 'a stub'}` still raises NotImplementedError. "
                  "Fill it in, then re-run this cell.")
        else:
            print(f"{label}: skipped, `{stub or 'a stub'}` is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED, {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"




## 1. The small platform

A small platform (the cell before the self-check prints how many users, agents, services and operations it has). All of it is in memory, and the clock is a number
you pass in: **nothing here reads the wall clock**, so every run is identical.

* **Users** sign in and get a *login token* whose audience is the token service itself (`"sts"`) and whose scopes are
  everything that user may do. `alice` may do everything; `bob` may only read the calendar.
* **The agent**, `agent-scribe`, is registered with the audiences it may ask for and the scopes it may hold. It is
  *narrower than any user*: it cannot send mail and it cannot write to the calendar.
* **Services** (`calendar-api`, `files-api`, `mail-api`) each own some operations, and each operation needs one scope.
  Destructive operations (`delete_file`, `send_mail`) are *single use*: one token, one call.

A token is three base64url parts joined by dots: a header (`alg`), the claims, and an HMAC-SHA256 signature over the
first two parts. `sign_token`, `peek_claims` and `_unb64` below are given; `peek_claims` reads a token **without verifying it**,
so use it for display and tests, or on a token you issued yourself, never on one you have not verified: an unverified token may not decode at all.

In [ ]:
T0 = 1_000_000                 # the simulated clock starts here, in seconds
DAY = 86_400
TTL = 300                      # the longest life of a token an agent is issued, in seconds
SUBJECT_TTL = 3_600            # the life of a user's login token, in seconds
STS = "sts"                    # audience of a login token: the token service itself
# A demo key derived from a string so every run is the same. A real key is random, secret and never in source.
SIGNING_KEY = hashlib.sha256(b"T07-L10 demo signing key: never derive a real key like this").digest()

USERS = {"alice": ("calendar:read", "calendar:write", "files:read", "files:write", "mail:send"),
         "bob": ("calendar:read",)}
AGENTS = {"agent-scribe": {"audiences": ("calendar-api", "files-api"),
                           "scopes": ("calendar:read", "files:read", "files:write")}}
AUDIENCES = ("calendar-api", "files-api", "mail-api")
OPS = {"read_events": ("calendar-api", "calendar:read"), "add_event": ("calendar-api", "calendar:write"),
       "read_file": ("files-api", "files:read"), "delete_file": ("files-api", "files:write"),
       "send_mail": ("mail-api", "mail:send")}          # operation -> (service that owns it, scope it needs)
SINGLE_USE = ("delete_file", "send_mail")


class Denied(Exception):
    """A request refused. `.code` is a short reason such as "expired"."""

    def __init__(self, code: str):
        super().__init__(code)
        self.code = code


def _b64(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).rstrip(b"=").decode()


def _unb64(text: str) -> bytes:
    """Decode base64url text, adding the padding it needs. Characters outside the base64url alphabet are skipped, not refused
    (`_unb64(" eyJh")` equals `_unb64("eyJh")`), so this raises ValueError only when the padding is then wrong or the text is not ASCII."""
    return base64.urlsafe_b64decode(text + "=" * (-len(text) % 4))


def _json(obj) -> bytes:
    return json.dumps(obj, sort_keys=True, separators=(",", ":")).encode()


def mac(signing_input: str, key: bytes) -> str:
    """The signature text for `signing_input` ("HEADER.PAYLOAD"): HMAC-SHA256, base64url without padding."""
    return _b64(hmac.new(key, signing_input.encode(), hashlib.sha256).digest())


def sign_token(claims: dict, key: bytes = SIGNING_KEY, alg: str = "HS256") -> str:
    """A signed compact token for `claims`. (`alg` is a parameter only so attacks can write a lying header.)"""
    head, body = _b64(_json({"alg": alg, "typ": "JWT"})), _b64(_json(claims))
    return f"{head}.{body}.{mac(head + '.' + body, key)}"


def peek_claims(token: str) -> dict:
    """The claims of a token, read WITHOUT verifying anything: for display and tests, or on a token you issued yourself, never on a
    token you have not verified (an unverified token may not even decode: this raises on "junk")."""
    return json.loads(_unb64(token.split(".")[1]))

## 2. What a login token holds

The next cell signs a login token for `alice` by hand and prints its claims. Read the claim names: `sub` is whose
authority it is, `aud` is who may accept it, `scope` is what it may do, `exp` is when it stops working, `jti` is its
unique id. The agent's token (section 4) will add one more, `act`, naming the agent.

In [ ]:
def show_login_token() -> None:
    login = sign_token({"iss": STS, "sub": "alice", "aud": STS, "scope": " ".join(sorted(USERS["alice"])),
                        "iat": T0, "exp": T0 + SUBJECT_TTL, "jti": "login-alice-0001"})
    head, body, sig = login.split(".")
    print("a token has", len(login.split(".")), "parts; the header says", json.loads(_unb64(head)))
    for name, value in peek_claims(login).items():
        print(f"  {name:6} {value}")
    print(f"a login token lives {SUBJECT_TTL} s on the simulated clock; an exchanged agent token lives at most {TTL} s")


if __name__ == "__main__":
    show_login_token()

## 3. Exercise 1: decide whether to believe a token

A bearer token is believed by whoever checks it, so the order of the checks is the whole design: **never read a claim
before the signature says the claims are the issuer's.** Write the checker every service in this lesson will use.

**`verify_token(token, key, now, audience, scope=None)`** returns the token's claims (a dict) or raises
`Denied(code)` for the *first* check that fails, in exactly this order:

1. `"malformed"`: `token` is not a string of exactly three dot-separated parts, or the first two parts are not base64url JSON objects (a JSON list, string or number is not an object; an empty `{}` is: see rules 2 and 4).
   Decode a part with the given `_unb64` (the base64url alphabet has `-` and `_`: never `base64.b64decode`), then `.decode("utf-8")` the bytes and parse the text with `json.loads`; a `ValueError` or a result that is not a dict is `"malformed"`, whatever the header says (a lying `alg` over a payload that does not decode is `"malformed"`).
   This lesson reads UTF-8 JSON only, strictly: a part written as UTF-16 or UTF-32, or as UTF-8 with a byte-order mark in front, or holding a byte that is not UTF-8 (never read with `errors="replace"`, `"ignore"` or as latin-1), is `"malformed"`; raw non-ASCII UTF-8 text is read as UTF-8 (a signed `sub` of `zoë` comes back as `zoë`). `_unb64` skips characters outside the alphabet, so a space before the header
   does not make a token malformed: it reaches the signature check;
2. `"bad-alg"`: the header's `alg` is not exactly `"HS256"` (a missing `alg` counts (a header of `{}` is `"bad-alg"`), and so does an empty one or a part of the name such as `"HS2"`; the header chooses nothing: `kid` and `typ`, whatever they hold, are ignored);
3. `"bad-signature"`: the third part is not exactly `mac("HEADER.PAYLOAD", key)`, computed over the first two parts
   exactly as received: never decode and re-encode them (a payload that is valid JSON spelled with extra spaces is
   checked over those spaces), and never strip, trim or normalise any part of the token (a newline or a space after
   the signature belongs to the signature text). Any other signature is bad whatever characters it holds (an appended
   `=`, another letter case or non-ASCII text such as `é` included). Compare in constant time with `hmac.compare_digest`:
   this is advice, because the checks here cannot see timing, only the answer;
4. `"bad-claims"` (before every claim is used, so before `"expired"`, `"not-yet-valid"`, `"wrong-audience"` and `"missing-scope"`, whichever claim has the wrong type and whatever else is wrong with the token): `sub`, `aud`, `jti` and `scope` are not all strings (whatever the `scope` argument is), or `exp` is not an `int`,
   or the key `nbf` is present and its value is not an `int` (null counts as present; a `bool` is not an `int`). An `int` of any sign is fine (`exp` of `0`
   or `-1` is `"expired"`, `nbf` of `-1` is satisfied). An `aud` list is bad here: this lesson's tokens name one audience; a signed payload of `{}` has none of them, so it is `"bad-claims"`;
5. `"expired"`: `now >= exp` (the token is good for `now < exp` and not a second longer);
6. `"not-yet-valid"`: `nbf` is present and `now < nbf`;
7. `"wrong-audience"`: `aud` is not exactly `audience` (case and every character count);
8. `"missing-scope"`: `scope` (the argument) is not `None` and is not one of the words in the claim `scope`, which you split
   with `str.split()` (OAuth uses single spaces; splitting on any whitespace is the lenient choice made here, so tabs, newlines and runs of them
   also separate words: so do the vertical tab, the form feed and the non-breaking space). It is a whole-word match, and case counts for scopes as it does for audiences: `files:readonly` does not give `files:read`, `Files:Read` and `FILES:READ` are not `files:read` (in the claim or in the argument), a required scope of two
   words, such as `"a b"`, is not one word of any claim, and `str.split()` leaves no empty word, so `""` matches no claim. The `scope` argument is compared exactly as it is given, never stripped or trimmed: `" files:read"`, `"files:read "` and `"files:read\n"` are not the word `files:read`.

The verifier does **not** look at `iat`, `iss` or the header's `typ`: a token read before its `iat`, or with no `iss`, no `iat` or
no `typ`, or with an `iss`, `iat` or `typ` of any type, is good if it passes the eight checks, and a token with no `nbf` has no
not-before time. An empty string is still a string, so an empty `sub` or `jti` is not `"bad-claims"`. Never change `token` or the claims.

<details><summary>💡 Hint 1 — what to think about</summary>

Decoding the token is not trusting it: the header and claims you decode first are still the sender's words until the
signature check has passed. Which of the eight reasons depend on a claim, and so must wait? What does the header's
`alg` let an attacker choose if you obey it?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Split on dots inside a `try` that turns every decoding problem into the first reason. Compare the header's `alg`
with the one allowed value. Recompute the signature with `mac` and compare it as bytes, constant time. Then test
the claims in the stated order. For the scope, split the claim on whitespace and test membership in the resulting list.

</details>

In [ ]:
def verify_token(token, key: bytes, now: int, audience: str, scope=None) -> dict:
    """Return the verified claims of `token`, or raise Denied with the first failing reason, as described above.

    Example:
        >>> token = sign_token({"sub": "alice", "aud": "files-api", "scope": "files:read", "exp": T0 + 60, "jti": "j"})
        >>> verify_token(token, SIGNING_KEY, T0, "files-api", "files:read")["sub"]
        'alice'
        >>> try:
        ...     verify_token(token, SIGNING_KEY, T0 + 60, "files-api")
        ... except Denied as exc:
        ...     exc.code
        'expired'

    Returns:
        The claims dict; Denied(code) when any check fails.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def make_claims(**over) -> dict:
    """Claims of a valid files-api token for the agent acting for alice; keyword arguments override a claim."""
    claims = {"iss": STS, "sub": "alice", "act": {"sub": "agent-scribe"}, "aud": "files-api", "scope": "files:read",
              "iat": T0, "exp": T0 + TTL, "jti": "j-1"}
    claims.update(over)
    return claims


def without(claims: dict, *names) -> dict:
    return {k: v for k, v in claims.items() if k not in names}


def _code(fn, *args, **kwargs):
    """What a call returns, or the reason it was denied as "denied:CODE"."""
    try:
        fn(*args, **kwargs)
    except Denied as exc:
        return f"denied:{exc.code}"
    return "allowed"


def _error_of(fn, *args):
    """The name of the exception a call raises, or "no error"."""
    try:
        fn(*args)
    except Exception as exc:
        return type(exc).__name__
    return "no error"


def _check_verify() -> None:
    key, now = SIGNING_KEY, T0 + 10

    def v(token, at=now, audience="files-api", scope="files:read", k=key):
        return _code(verify_token, token, k, at, audience, scope)

    good = sign_token(make_claims())
    head, body, sig = good.split(".")
    got = verify_token(good, key, now, "files-api", "files:read")
    assert got == make_claims(), 'a good token returns its claims, exactly as signed'
    assert isinstance(got, dict), 'a good token returns a plain dict, not a read-only view or another mapping'
    assert v(good, scope=None) == "allowed", "scope=None means no scope is required"
    zdump = dict(sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    zbody = _b64(json.dumps(make_claims(sub="zoë"), **zdump).encode("utf-8"))
    zbody2 = _b64(json.dumps(make_claims(sub="zoë", jti="j-2"), **zdump).encode("utf-8"))
    assert v(f"{head}.{zbody}.{mac(head + '.' + zbody, key)}") == "allowed", "a payload with raw non-ASCII UTF-8 text (a sub of 'zoë', signed as written) is a good token: decode the bytes as UTF-8 (not as ASCII)"
    assert verify_token(f"{head}.{zbody2}.{mac(head + '.' + zbody2, key)}", key, now, "files-api", "files:read")["sub"] == "zoë", "raw UTF-8 text comes back as it was written: decode the bytes as UTF-8, never as latin-1 (a sub of 'zoë' is not 'zoÃ«')"
    assert v('') == 'denied:malformed', "'': not three dot-separated parts is 'malformed' (and None or a number must not crash)"
    assert v('x') == 'denied:malformed', "'x': not three dot-separated parts is 'malformed' (and None or a number must not crash)"
    assert v('abc') == 'denied:malformed', "'abc': not three dot-separated parts is 'malformed' (and None or a number must not crash)"
    assert v('a.b') == 'denied:malformed', "'a.b': not three dot-separated parts is 'malformed' (and None or a number must not crash)"
    assert v(f'{good}.x') == 'denied:malformed', f"{f'{good}.x'!r}: not three dot-separated parts is 'malformed' (and None or a number must not crash)"
    assert v(None) == 'denied:malformed', "None: not three dot-separated parts is 'malformed' (and None or a number must not crash)"
    assert v(12) == 'denied:malformed', "12: not three dot-separated parts is 'malformed' (and None or a number must not crash)"
    assert v(3.5) == 'denied:malformed', "3.5: not three dot-separated parts is 'malformed' (and None or a number must not crash)"
    assert v(b'a.b.c') == 'denied:malformed', "b'a.b.c': not three dot-separated parts is 'malformed' (and None or a number must not crash)"
    assert v(b'h.b.s') == 'denied:malformed', "b'h.b.s': not three dot-separated parts is 'malformed' (and None or a number must not crash)"
    assert v(['a', 'b', 'c']) == 'denied:malformed', "['a', 'b', 'c']: not three dot-separated parts is 'malformed' (and None or a number must not crash)"
    assert v(good.replace('.', '', 1)) == 'denied:malformed', f"{good.replace('.', '', 1)!r}: not three dot-separated parts is 'malformed' (and None or a number must not crash)"
    assert v(f"{head}.{_b64(b'nope')}.{sig}") == 'denied:malformed', f"a payload of {_b64(b'nope')!r} is not a base64url JSON object: 'malformed'"
    assert v(f"{_b64(b'nope')}.{body}.{sig}") == 'denied:malformed', f"a header of {_b64(b'nope')!r} is not a base64url JSON object: 'malformed' too"
    assert v(f"{head}.{_b64(b'{')}.{sig}") == 'denied:malformed', f"a payload of {_b64(b'{')!r} is not a base64url JSON object: 'malformed'"
    assert v(f"{_b64(b'{')}.{body}.{sig}") == 'denied:malformed', f"a header of {_b64(b'{')!r} is not a base64url JSON object: 'malformed' too"
    assert v(f"{head}.{_b64(b'[]')}.{sig}") == 'denied:malformed', f"a payload of {_b64(b'[]')!r} is not a base64url JSON object: 'malformed'"
    assert v(f"{_b64(b'[]')}.{body}.{sig}") == 'denied:malformed', f"a header of {_b64(b'[]')!r} is not a base64url JSON object: 'malformed' too"
    assert v(f"{head}.{_b64(b'[1]')}.{sig}") == 'denied:malformed', f"a payload of {_b64(b'[1]')!r} is not a base64url JSON object: 'malformed'"
    assert v(f"{_b64(b'[1]')}.{body}.{sig}") == 'denied:malformed', f"a header of {_b64(b'[1]')!r} is not a base64url JSON object: 'malformed' too"
    assert v(f"""{head}.{_b64(b'"x"')}.{sig}""") == 'denied:malformed', f"""a payload of {_b64(b'"x"')!r} is not a base64url JSON object: 'malformed'"""
    assert v(f"""{_b64(b'"x"')}.{body}.{sig}""") == 'denied:malformed', f"""a header of {_b64(b'"x"')!r} is not a base64url JSON object: 'malformed' too"""
    assert v(f'{head}.!!!!.{sig}') == 'denied:malformed', "a payload of '!!!!' is not a base64url JSON object: 'malformed'"
    assert v(f'!!!!.{body}.{sig}') == 'denied:malformed', "a header of '!!!!' is not a base64url JSON object: 'malformed' too"
    assert v(f'{head}.???.{sig}') == 'denied:malformed', "a payload of '???' is not a base64url JSON object: 'malformed'"
    assert v(f'???.{body}.{sig}') == 'denied:malformed', "a header of '???' is not a base64url JSON object: 'malformed' too"
    assert v(f"{head}.{_b64(b'123')}.{sig}") == 'denied:malformed', f"a payload of {_b64(b'123')!r} is not a base64url JSON object: 'malformed'"
    assert v(f"{_b64(b'123')}.{body}.{sig}") == 'denied:malformed', f"a header of {_b64(b'123')!r} is not a base64url JSON object: 'malformed' too"
    assert v(f"{head}.{_b64(b'null')}.{sig}") == 'denied:malformed', f"a payload of {_b64(b'null')!r} is not a base64url JSON object: 'malformed'"
    assert v(f"{_b64(b'null')}.{body}.{sig}") == 'denied:malformed', f"a header of {_b64(b'null')!r} is not a base64url JSON object: 'malformed' too"
    assert v(str(head) + '.' + str(_b64(b'\xff\xfe')) + '.' + str(sig)) == 'denied:malformed', 'a payload of ' + repr(_b64(b'\xff\xfe')) + " is not a base64url JSON object: 'malformed'"
    assert v(str(_b64(b'\xff\xfe')) + '.' + str(body) + '.' + str(sig)) == 'denied:malformed', 'a header of ' + repr(_b64(b'\xff\xfe')) + " is not a base64url JSON object: 'malformed' too"
    wide_head = _b64(_json({'alg': 'HS256', 'typ': 'JWT'}).decode().encode('utf-8-sig'))
    wide_body = _b64(_json(make_claims()).decode().encode('utf-8-sig'))
    assert v(f"{wide_head}.{body}.{mac(wide_head + '.' + body, key)}") == 'denied:malformed', "a header written as utf-8-sig is not UTF-8 JSON, though its signature is good: 'malformed' (decode the bytes as UTF-8 before json.loads: json.loads on bytes would sniff UTF-16 and UTF-32 and skip a UTF-8 byte-order mark)"
    assert v(f"{head}.{wide_body}.{mac(head + '.' + wide_body, key)}") == 'denied:malformed', "a payload written as utf-8-sig is not UTF-8 JSON, though its signature is good: 'malformed' (decode the bytes as UTF-8 before json.loads)"
    wide_head = _b64(_json({'alg': 'HS256', 'typ': 'JWT'}).decode().encode('utf-16'))
    wide_body = _b64(_json(make_claims()).decode().encode('utf-16'))
    assert v(f"{wide_head}.{body}.{mac(wide_head + '.' + body, key)}") == 'denied:malformed', "a header written as utf-16 is not UTF-8 JSON, though its signature is good: 'malformed' (decode the bytes as UTF-8 before json.loads: json.loads on bytes would sniff UTF-16 and UTF-32 and skip a UTF-8 byte-order mark)"
    assert v(f"{head}.{wide_body}.{mac(head + '.' + wide_body, key)}") == 'denied:malformed', "a payload written as utf-16 is not UTF-8 JSON, though its signature is good: 'malformed' (decode the bytes as UTF-8 before json.loads)"
    wide_head = _b64(_json({'alg': 'HS256', 'typ': 'JWT'}).decode().encode('utf-16-le'))
    wide_body = _b64(_json(make_claims()).decode().encode('utf-16-le'))
    assert v(f"{wide_head}.{body}.{mac(wide_head + '.' + body, key)}") == 'denied:malformed', "a header written as utf-16-le is not UTF-8 JSON, though its signature is good: 'malformed' (decode the bytes as UTF-8 before json.loads: json.loads on bytes would sniff UTF-16 and UTF-32 and skip a UTF-8 byte-order mark)"
    assert v(f"{head}.{wide_body}.{mac(head + '.' + wide_body, key)}") == 'denied:malformed', "a payload written as utf-16-le is not UTF-8 JSON, though its signature is good: 'malformed' (decode the bytes as UTF-8 before json.loads)"
    wide_head = _b64(_json({'alg': 'HS256', 'typ': 'JWT'}).decode().encode('utf-16-be'))
    wide_body = _b64(_json(make_claims()).decode().encode('utf-16-be'))
    assert v(f"{wide_head}.{body}.{mac(wide_head + '.' + body, key)}") == 'denied:malformed', "a header written as utf-16-be is not UTF-8 JSON, though its signature is good: 'malformed' (decode the bytes as UTF-8 before json.loads: json.loads on bytes would sniff UTF-16 and UTF-32 and skip a UTF-8 byte-order mark)"
    assert v(f"{head}.{wide_body}.{mac(head + '.' + wide_body, key)}") == 'denied:malformed', "a payload written as utf-16-be is not UTF-8 JSON, though its signature is good: 'malformed' (decode the bytes as UTF-8 before json.loads)"
    wide_head = _b64(_json({'alg': 'HS256', 'typ': 'JWT'}).decode().encode('utf-32'))
    wide_body = _b64(_json(make_claims()).decode().encode('utf-32'))
    assert v(f"{wide_head}.{body}.{mac(wide_head + '.' + body, key)}") == 'denied:malformed', "a header written as utf-32 is not UTF-8 JSON, though its signature is good: 'malformed' (decode the bytes as UTF-8 before json.loads: json.loads on bytes would sniff UTF-16 and UTF-32 and skip a UTF-8 byte-order mark)"
    assert v(f"{head}.{wide_body}.{mac(head + '.' + wide_body, key)}") == 'denied:malformed', "a payload written as utf-32 is not UTF-8 JSON, though its signature is good: 'malformed' (decode the bytes as UTF-8 before json.loads)"
    wide_head = _b64(_json({'alg': 'HS256', 'typ': 'JWT'}).decode().encode('utf-32-le'))
    wide_body = _b64(_json(make_claims()).decode().encode('utf-32-le'))
    assert v(f"{wide_head}.{body}.{mac(wide_head + '.' + body, key)}") == 'denied:malformed', "a header written as utf-32-le is not UTF-8 JSON, though its signature is good: 'malformed' (decode the bytes as UTF-8 before json.loads: json.loads on bytes would sniff UTF-16 and UTF-32 and skip a UTF-8 byte-order mark)"
    assert v(f"{head}.{wide_body}.{mac(head + '.' + wide_body, key)}") == 'denied:malformed', "a payload written as utf-32-le is not UTF-8 JSON, though its signature is good: 'malformed' (decode the bytes as UTF-8 before json.loads)"
    wide_head = _b64(_json({'alg': 'HS256', 'typ': 'JWT'}).decode().encode('utf-32-be'))
    wide_body = _b64(_json(make_claims()).decode().encode('utf-32-be'))
    assert v(f"{wide_head}.{body}.{mac(wide_head + '.' + body, key)}") == 'denied:malformed', "a header written as utf-32-be is not UTF-8 JSON, though its signature is good: 'malformed' (decode the bytes as UTF-8 before json.loads: json.loads on bytes would sniff UTF-16 and UTF-32 and skip a UTF-8 byte-order mark)"
    assert v(f"{head}.{wide_body}.{mac(head + '.' + wide_body, key)}") == 'denied:malformed', "a payload written as utf-32-be is not UTF-8 JSON, though its signature is good: 'malformed' (decode the bytes as UTF-8 before json.loads)"
    raw_head = _b64(b'{"alg":"HS256","kid":"\xff","typ":"JWT"}')
    raw_body = _b64(_json(make_claims()).replace(b'"j-1"', b'"j\xff1"'))
    assert v(f"{raw_head}.{body}.{mac(raw_head + '.' + body, key)}") == 'denied:malformed', "a header holding a byte that is not UTF-8 (0xff inside a string) is not UTF-8 JSON, though its signature is good: 'malformed' (decode strictly: errors='replace', errors='ignore' and latin-1 would read it)"
    assert v(f"{head}.{raw_body}.{mac(head + '.' + raw_body, key)}") == 'denied:malformed', "a payload holding a byte that is not UTF-8 (0xff inside a string) is not UTF-8 JSON, though its signature is good: 'malformed' (decode strictly: errors='replace', errors='ignore' and latin-1 would read it)"
    lying = _b64(_json({"alg": "none", "typ": "JWT"}))
    assert v(f"{lying}.{_b64(b'nope')}.{sig}") == 'denied:malformed', f"a header that lies about alg over a payload of {_b64(b'nope')!r} is 'malformed': both parts are decoded and checked before any other reason, bad-alg included"
    assert v(f"{lying}.{_b64(b'[]')}.{sig}") == 'denied:malformed', f"a header that lies about alg over a payload of {_b64(b'[]')!r} is 'malformed': both parts are decoded and checked before any other reason, bad-alg included"
    assert v(f'{lying}.!!!!.{sig}') == 'denied:malformed', "a header that lies about alg over a payload of '!!!!' is 'malformed': both parts are decoded and checked before any other reason, bad-alg included"
    assert v(sign_token(make_claims(), alg="none")) == "denied:bad-alg", (
        "this token carries a VALID signature but a header that says alg none: refuse by header, whatever the signature")
    assert v(sign_token(make_claims(), alg="HS512")) == "denied:bad-alg", "only HS256 is allowed"
    assert v(sign_token(make_claims(), alg="RS256")) == "denied:bad-alg", "only HS256 is allowed: an algorithm that needs another kind of key is refused too"
    assert v(sign_token(make_claims(), alg="hs256")) == "denied:bad-alg", "the algorithm name is compared exactly, case included"
    assert v(sign_token(make_claims(), alg="HS256 ")) == "denied:bad-alg", "the algorithm name is compared exactly, a trailing space included (no strip, no normalising)"
    assert v(sign_token(make_claims(), alg='')) == 'denied:bad-alg', "an alg of '' is not exactly 'HS256': compare the whole value, not a part of it (a test such as `alg in 'HS256'` is a substring test)"
    assert v(sign_token(make_claims(), alg='HS')) == 'denied:bad-alg', "an alg of 'HS' is not exactly 'HS256': compare the whole value, not a part of it (a test such as `alg in 'HS256'` is a substring test)"
    assert v(sign_token(make_claims(), alg='HS2')) == 'denied:bad-alg', "an alg of 'HS2' is not exactly 'HS256': compare the whole value, not a part of it (a test such as `alg in 'HS256'` is a substring test)"
    assert v(sign_token(make_claims(), alg='256')) == 'denied:bad-alg', "an alg of '256' is not exactly 'HS256': compare the whole value, not a part of it (a test such as `alg in 'HS256'` is a substring test)"
    assert v(sign_token(make_claims(), alg='S256')) == 'denied:bad-alg', "an alg of 'S256' is not exactly 'HS256': compare the whole value, not a part of it (a test such as `alg in 'HS256'` is a substring test)"
    assert v(sign_token(make_claims(), alg=5)) == 'denied:bad-alg', "an alg of 5 is not exactly 'HS256': compare the whole value, not a part of it (a test such as `alg in 'HS256'` is a substring test)"
    assert v(sign_token(make_claims(), alg=['HS256'])) == 'denied:bad-alg', "an alg of ['HS256'] is not exactly 'HS256': compare the whole value, not a part of it (a test such as `alg in 'HS256'` is a substring test)"
    assert v(sign_token(make_claims(exp=T0 + 1), alg="none")) == "denied:bad-alg", "bad-alg comes before every claim check: a lying header on an expired token is 'bad-alg'"
    assert v(sign_token(make_claims(), key=b"another key", alg="none")) == "denied:bad-alg", "bad-alg comes before the signature check: a lying header AND a wrong key is 'bad-alg'"
    assert v(f"{head}.{body}.") == "denied:bad-signature", "an empty signature on an HS256 header is 'bad-signature'"
    unsigned = _b64(_json({"typ": "JWT"})) + f".{body}."
    assert v(unsigned) == "denied:bad-alg", "a header with no alg at all is 'bad-alg'"
    empty_head = _b64(_json({}))
    assert v(f"{empty_head}.{body}.{mac(empty_head + '.' + body, key)}") == "denied:bad-alg", (
        "a header of {} is a JSON object with no alg: 'bad-alg', not 'malformed' (an empty dict is falsy, but it is still an object)")
    assert v(f"{empty_head}.{body}.") == "denied:bad-alg", "a header of {} is 'bad-alg' before the signature is looked at"
    assert v(good, k=b"another key") == "denied:bad-signature", "a token signed with another key is 'bad-signature'"
    edited = f"{head}.{_b64(_json(make_claims(scope='files:read files:write')))}.{sig}"
    assert v(edited) == "denied:bad-signature", "edited claims under the old signature are 'bad-signature'"
    assert v(f"{head}.{body}.{sig[:-1]}{'A' if sig[-1] != 'A' else 'B'}") == "denied:bad-signature", "one changed signature character is enough"
    assert v(f"{head}.{body}.é") == "denied:bad-signature", "a signature of any characters, non-ASCII included, is simply bad (compare_digest on str raises for non-ASCII)"
    assert v(f'{head}.{body}.ü') == 'denied:bad-signature', "signature 'ü': the third part must equal the expected text exactly: no padding added, no case folding, no decoding to bytes first, nothing ignored"
    assert v(f'{head}.{body}.€€') == 'denied:bad-signature', "signature '€€': the third part must equal the expected text exactly: no padding added, no case folding, no decoding to bytes first, nothing ignored"
    assert v(f"{head}.{body}.{f'{sig}='}") == 'denied:bad-signature', f"signature {f'{sig}='!r}: the third part must equal the expected text exactly: no padding added, no case folding, no decoding to bytes first, nothing ignored"
    assert v(f"{head}.{body}.{f'{sig}{sig}'}") == 'denied:bad-signature', f"signature {f'{sig}{sig}'!r}: the third part must equal the expected text exactly: no padding added, no case folding, no decoding to bytes first, nothing ignored"
    assert v(f"{head}.{body}.{f'{sig}é'}") == 'denied:bad-signature', f"signature {f'{sig}é'!r}: the third part must equal the expected text exactly: no padding added, no case folding, no decoding to bytes first, nothing ignored"
    assert v(f"{head}.{body}.{f'{sig}€'}") == 'denied:bad-signature', f"signature {f'{sig}€'!r}: the third part must equal the expected text exactly: no padding added, no case folding, no decoding to bytes first, nothing ignored"
    assert v(f'{head}.{body}.{(sig.upper() if sig.upper() != sig else sig.lower())}') == 'denied:bad-signature', f'signature {(sig.upper() if sig.upper() != sig else sig.lower())!r}: the third part must equal the expected text exactly: no padding added, no case folding, no decoding to bytes first, nothing ignored'
    spaced_body = _b64(json.dumps(make_claims()).encode())              # the same claims, spelled with the spaces json.dumps adds
    assert v(f"{head}.{spaced_body}.{mac(f'{head}.{spaced_body}', key)}") == "allowed", (
        "a token signed over the text it arrived in is good, however its JSON is spelled: check the signature over the first two parts AS RECEIVED, never over a re-encoding")
    assert v(f"{head}.{spaced_body}.{sig}") == "denied:bad-signature", (
        "the signature of the compact text does not cover this spaced text: compute the signature over the received parts, not over claims you decoded and encoded again")
    assert v(f'{good}\n') == 'denied:bad-signature', repr(f'{good}\n') + ': a space or newline after the token belongs to the signature text: do not strip or trim anything before the checks'
    assert v(f'{good} ') == 'denied:bad-signature', f"{f'{good} '!r}: a space or newline after the token belongs to the signature text: do not strip or trim anything before the checks"
    assert v(f" {good}") == "denied:bad-signature", (
        "a space before the header is not stripped, and `_unb64` still decodes the header, so the token is not malformed: the signature check, over the text as received, refuses it")
    stale_edit = f"{head}.{_b64(_json(make_claims(exp=T0 + 1)))}.{sig}"
    assert v(stale_edit) == "denied:bad-signature", "the signature is checked BEFORE any claim is read: edited and expired is 'bad-signature'"
    odd_edit = f"{head}.{_b64(_json(make_claims(sub=9, aud='mail-api')))}.{sig}"
    assert v(odd_edit) == "denied:bad-signature", "the signature is checked before the claims' types too: edited claims of the wrong type are 'bad-signature'"
    assert v(sign_token(without(make_claims(), 'sub'))) == 'denied:bad-claims', "a token without sub is 'bad-claims'"
    assert v(sign_token(without(make_claims(), 'aud'))) == 'denied:bad-claims', "a token without aud is 'bad-claims'"
    assert v(sign_token(without(make_claims(), 'jti'))) == 'denied:bad-claims', "a token without jti is 'bad-claims'"
    assert v(sign_token(without(make_claims(), 'exp'))) == 'denied:bad-claims', "a token without exp is 'bad-claims'"
    assert v(sign_token(without(make_claims(), 'scope'))) == 'denied:bad-claims', "a token without scope is 'bad-claims'"
    assert v(sign_token({}), scope=None) == 'denied:bad-claims', "a signed payload of {} (scope argument None) is a JSON object with none of the claims: 'bad-claims', not 'malformed'"
    assert v(sign_token({}), scope='files:read') == 'denied:bad-claims', "a signed payload of {} (scope argument 'files:read') is a JSON object with none of the claims: 'bad-claims', not 'malformed'"
    assert v(sign_token(make_claims(exp='9999999'))) == 'denied:bad-claims', "{'exp': '9999999'}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(exp=float(T0 + 500)))) == 'denied:bad-claims', "{'exp': 1000500.0}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(exp=True))) == 'denied:bad-claims', "{'exp': True}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(exp=False))) == 'denied:bad-claims', "{'exp': False}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(exp=None))) == 'denied:bad-claims', "{'exp': None}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(aud=['files-api']))) == 'denied:bad-claims', "{'aud': ['files-api']}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(aud=5))) == 'denied:bad-claims', "{'aud': 5}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(nbf='5'))) == 'denied:bad-claims', "{'nbf': '5'}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(nbf=None))) == 'denied:bad-claims', "{'nbf': None}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(nbf=True))) == 'denied:bad-claims', "{'nbf': True}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(nbf=False))) == 'denied:bad-claims', "{'nbf': False}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(nbf=1.0))) == 'denied:bad-claims', "{'nbf': 1.0}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(nbf=[1]))) == 'denied:bad-claims', "{'nbf': [1]}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(sub=7))) == 'denied:bad-claims', "{'sub': 7}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(sub=None))) == 'denied:bad-claims', "{'sub': None}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(jti=1))) == 'denied:bad-claims', "{'jti': 1}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(scope=['files:read']))) == 'denied:bad-claims', "{'scope': ['files:read']}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(make_claims(scope=None))) == 'denied:bad-claims', "{'scope': None}: wrong types are 'bad-claims' (a bool is not an int, and a float is not one either)"
    assert v(sign_token(without(make_claims(), 'sub')), scope=None) == 'denied:bad-claims', "a token without sub is 'bad-claims' whatever the scope argument is, None included"
    assert v(sign_token(without(make_claims(), 'aud')), scope=None) == 'denied:bad-claims', "a token without aud is 'bad-claims' whatever the scope argument is, None included"
    assert v(sign_token(without(make_claims(), 'jti')), scope=None) == 'denied:bad-claims', "a token without jti is 'bad-claims' whatever the scope argument is, None included"
    assert v(sign_token(without(make_claims(), 'exp')), scope=None) == 'denied:bad-claims', "a token without exp is 'bad-claims' whatever the scope argument is, None included"
    assert v(sign_token(without(make_claims(), 'scope')), scope=None) == 'denied:bad-claims', "a token without scope is 'bad-claims' whatever the scope argument is, None included"
    assert v(sign_token(make_claims(scope=None)), scope=None) == 'denied:bad-claims', "{'scope': None}: wrong types are 'bad-claims' with scope=None too: the claims are checked whether or not a scope is required"
    assert v(sign_token(make_claims(scope=5)), scope=None) == 'denied:bad-claims', "{'scope': 5}: wrong types are 'bad-claims' with scope=None too: the claims are checked whether or not a scope is required"
    assert v(sign_token(make_claims(scope=['files:read'])), scope=None) == 'denied:bad-claims', "{'scope': ['files:read']}: wrong types are 'bad-claims' with scope=None too: the claims are checked whether or not a scope is required"
    assert v(sign_token(make_claims(sub=7)), scope=None) == 'denied:bad-claims', "{'sub': 7}: wrong types are 'bad-claims' with scope=None too: the claims are checked whether or not a scope is required"
    assert v(sign_token(make_claims(nbf='5')), scope=None) == 'denied:bad-claims', "{'nbf': '5'}: wrong types are 'bad-claims' with scope=None too: the claims are checked whether or not a scope is required"
    assert v(sign_token(make_claims(exp="x")), audience="mail-api") == "denied:bad-claims", "claim types are checked before audience"
    assert v(sign_token(make_claims(sub=7, exp=T0 + 1))) == "denied:bad-claims", "claim types are checked before expiry"
    # every typed claim, with the wrong type, together with each other fault a token can have: bad-claims comes first, for each claim
    # (a type check made only where the claim is used, such as nbf at the not-before step or aud at the audience step, gives another reason)
    assert v(sign_token(make_claims(exp=T0 + 1, sub=7)), audience='files-api', scope=None) == 'denied:bad-claims', "sub = 7 together with expired (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(sub=7)), audience='mail-api', scope=None) == 'denied:bad-claims', "sub = 7 together with the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, sub=7)), audience='mail-api', scope=None) == 'denied:bad-claims', "sub = 7 together with expired and the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, sub=7)), audience='files-api', scope=None) == 'denied:bad-claims', "sub = 7 together with not yet valid (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, sub=7)), audience='mail-api', scope=None) == 'denied:bad-claims', "sub = 7 together with not yet valid and the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(sub=7)), audience='mail-api', scope='files:read') == 'denied:bad-claims', "sub = 7 together with the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, sub=7)), audience='mail-api', scope='files:read') == 'denied:bad-claims', "sub = 7 together with expired and the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, sub=7)), audience='files-api', scope='files:read') == 'denied:bad-claims', "sub = 7 together with not yet valid (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, sub=7)), audience='mail-api', scope='files:read') == 'denied:bad-claims', "sub = 7 together with not yet valid and the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(scope='calendar:read', sub=7)), audience='files-api', scope='files:read') == 'denied:bad-claims', "sub = 7 together with a scope it lacks (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, aud=5)), audience='files-api', scope=None) == 'denied:bad-claims', "aud = 5 together with expired (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(aud=5)), audience='mail-api', scope=None) == 'denied:bad-claims', "aud = 5 together with the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, aud=5)), audience='mail-api', scope=None) == 'denied:bad-claims', "aud = 5 together with expired and the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, aud=5)), audience='files-api', scope=None) == 'denied:bad-claims', "aud = 5 together with not yet valid (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, aud=5)), audience='mail-api', scope=None) == 'denied:bad-claims', "aud = 5 together with not yet valid and the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, aud=5)), audience='files-api', scope='files:read') == 'denied:bad-claims', "aud = 5 together with expired (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(aud=5)), audience='mail-api', scope='files:read') == 'denied:bad-claims', "aud = 5 together with the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, aud=5)), audience='mail-api', scope='files:read') == 'denied:bad-claims', "aud = 5 together with expired and the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, aud=5)), audience='files-api', scope='files:read') == 'denied:bad-claims', "aud = 5 together with not yet valid (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, aud=5)), audience='mail-api', scope='files:read') == 'denied:bad-claims', "aud = 5 together with not yet valid and the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(scope='calendar:read', aud=5)), audience='files-api', scope='files:read') == 'denied:bad-claims', "aud = 5 together with a scope it lacks (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, jti=1)), audience='files-api', scope=None) == 'denied:bad-claims', "jti = 1 together with expired (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(jti=1)), audience='mail-api', scope=None) == 'denied:bad-claims', "jti = 1 together with the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, jti=1)), audience='mail-api', scope=None) == 'denied:bad-claims', "jti = 1 together with expired and the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, jti=1)), audience='files-api', scope=None) == 'denied:bad-claims', "jti = 1 together with not yet valid (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, jti=1)), audience='mail-api', scope=None) == 'denied:bad-claims', "jti = 1 together with not yet valid and the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, jti=1)), audience='files-api', scope='files:read') == 'denied:bad-claims', "jti = 1 together with expired (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(jti=1)), audience='mail-api', scope='files:read') == 'denied:bad-claims', "jti = 1 together with the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, jti=1)), audience='mail-api', scope='files:read') == 'denied:bad-claims', "jti = 1 together with expired and the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, jti=1)), audience='files-api', scope='files:read') == 'denied:bad-claims', "jti = 1 together with not yet valid (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, jti=1)), audience='mail-api', scope='files:read') == 'denied:bad-claims', "jti = 1 together with not yet valid and the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(scope='calendar:read', jti=1)), audience='files-api', scope='files:read') == 'denied:bad-claims', "jti = 1 together with a scope it lacks (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, scope=None)), audience='files-api', scope=None) == 'denied:bad-claims', "scope = None together with expired (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(scope=None)), audience='mail-api', scope=None) == 'denied:bad-claims', "scope = None together with the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, scope=None)), audience='mail-api', scope=None) == 'denied:bad-claims', "scope = None together with expired and the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, scope=None)), audience='files-api', scope=None) == 'denied:bad-claims', "scope = None together with not yet valid (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, scope=None)), audience='mail-api', scope=None) == 'denied:bad-claims', "scope = None together with not yet valid and the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, scope=None)), audience='files-api', scope='files:read') == 'denied:bad-claims', "scope = None together with expired (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(scope=None)), audience='mail-api', scope='files:read') == 'denied:bad-claims', "scope = None together with the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, scope=None)), audience='mail-api', scope='files:read') == 'denied:bad-claims', "scope = None together with expired and the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, scope=None)), audience='files-api', scope='files:read') == 'denied:bad-claims', "scope = None together with not yet valid (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, scope=None)), audience='mail-api', scope='files:read') == 'denied:bad-claims', "scope = None together with not yet valid and the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp='x')), audience='mail-api', scope=None) == 'denied:bad-claims', "exp = 'x' together with the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, exp='x')), audience='files-api', scope=None) == 'denied:bad-claims', "exp = 'x' together with not yet valid (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, exp='x')), audience='mail-api', scope=None) == 'denied:bad-claims', "exp = 'x' together with not yet valid and the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, exp='x')), audience='files-api', scope='files:read') == 'denied:bad-claims', "exp = 'x' together with not yet valid (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=T0 + 500, exp='x')), audience='mail-api', scope='files:read') == 'denied:bad-claims', "exp = 'x' together with not yet valid and the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(scope='calendar:read', exp='x')), audience='files-api', scope='files:read') == 'denied:bad-claims', "exp = 'x' together with a scope it lacks (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, nbf='5')), audience='files-api', scope=None) == 'denied:bad-claims', "nbf = '5' together with expired (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf='5')), audience='mail-api', scope=None) == 'denied:bad-claims', "nbf = '5' together with the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, nbf='5')), audience='mail-api', scope=None) == 'denied:bad-claims', "nbf = '5' together with expired and the wrong audience (required scope None): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, nbf='5')), audience='files-api', scope='files:read') == 'denied:bad-claims', "nbf = '5' together with expired (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf='5')), audience='mail-api', scope='files:read') == 'denied:bad-claims', "nbf = '5' together with the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(exp=T0 + 1, nbf='5')), audience='mail-api', scope='files:read') == 'denied:bad-claims', "nbf = '5' together with expired and the wrong audience (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(scope='calendar:read', nbf='5')), audience='files-api', scope='files:read') == 'denied:bad-claims', "nbf = '5' together with a scope it lacks (required scope 'files:read'): the type of every claim is checked before any of them is used, so this is 'bad-claims'"
    assert v(sign_token(make_claims(nbf=0))) == "allowed", "an nbf of 0 is a valid int"
    exp = make_claims()["exp"]
    assert v(sign_token(make_claims()), at=exp - 1) == "allowed", "one second before exp is still good"
    assert v(sign_token(make_claims()), at=exp) == "denied:expired", "at exp the token is expired: now >= exp"
    assert v(sign_token(make_claims()), at=exp + DAY) == "denied:expired", "a day after exp the token is still expired: now >= exp has no upper end"
    early = sign_token(make_claims(nbf=T0 + 50))
    assert v(early, at=T0 + 49) == "denied:not-yet-valid", "nbf: refused while now < nbf"
    assert v(early, at=T0 + 50) == "allowed", "nbf: fine from now == nbf on"
    assert v(sign_token(make_claims(nbf=T0 + 100, exp=T0 + 20)), at=T0 + 30) == "denied:expired", "expiry is tested before not-before"
    assert v(sign_token(make_claims(nbf=T0 + 500)), audience="mail-api") == "denied:not-yet-valid", "not-before is tested before audience: a token that is both early and for another service is 'not-yet-valid'"
    assert v(sign_token(make_claims()), at=T0 - 5) == "allowed", "iat is not checked: a token with no nbf is good before its iat too"
    assert v(good, audience='mail-api') == 'denied:wrong-audience', "audience 'mail-api' is not exactly files-api"
    assert v(good, audience='Files-API') == 'denied:wrong-audience', "audience 'Files-API' is not exactly files-api"
    assert v(good, audience='files-api-2') == 'denied:wrong-audience', "audience 'files-api-2' is not exactly files-api"
    assert v(good, audience='files-api ') == 'denied:wrong-audience', "audience 'files-api ' is not exactly files-api"
    assert v(good, audience=' files-api') == 'denied:wrong-audience', "audience ' files-api' is not exactly files-api"
    assert v(good, audience='files-api/') == 'denied:wrong-audience', "audience 'files-api/' is not exactly files-api"
    assert v(good, audience='files') == 'denied:wrong-audience', "audience 'files' is not exactly files-api"
    assert v(good, audience='api') == 'denied:wrong-audience', "audience 'api' is not exactly files-api"
    assert v(good, audience='') == 'denied:wrong-audience', "audience '' is not exactly files-api"
    assert v(sign_token(make_claims(exp=T0 + 5)), audience="mail-api") == "denied:expired", "expiry is tested before audience"
    assert v(good, audience="mail-api", scope="mail:send") == "denied:wrong-audience", "audience is tested before scope"
    multi = sign_token(make_claims(scope="calendar:read files:read files:write"))
    assert v(multi, scope="files:read") == "allowed", "the scope claim is a space-separated list: a word in the middle"
    assert v(multi, scope="files:write") == "allowed", "the scope claim is a space-separated list: the last word"
    assert v(sign_token(make_claims(scope="files:readonly")), scope="files:read") == "denied:missing-scope", "a whole-word match, not a prefix or substring"
    near = sign_token(make_claims(scope="files:readonly calendar:read"))
    assert v(near, scope='files:read') == 'denied:missing-scope', "'files:read' is not one whole word of the scope claim: a prefix, a substring or a set of words is wrong"
    assert v(near, scope='files') == 'denied:missing-scope', "'files' is not one whole word of the scope claim: a prefix, a substring or a set of words is wrong"
    assert v(near, scope='read') == 'denied:missing-scope', "'read' is not one whole word of the scope claim: a prefix, a substring or a set of words is wrong"
    assert v(near, scope='calendar') == 'denied:missing-scope', "'calendar' is not one whole word of the scope claim: a prefix, a substring or a set of words is wrong"
    assert v(near, scope='files:read calendar:read') == 'denied:missing-scope', "'files:read calendar:read' is not one whole word of the scope claim: a prefix, a substring or a set of words is wrong"
    assert v(near, scope='files:readonly calendar') == 'denied:missing-scope', "'files:readonly calendar' is not one whole word of the scope claim: a prefix, a substring or a set of words is wrong"
    assert v(near, scope='') == 'denied:missing-scope', "'' is not one whole word of the scope claim: a prefix, a substring or a set of words is wrong"
    assert v(near, scope='files:readonly') == 'allowed', 'files:readonly is one of the words of the scope claim'
    assert v(near, scope='calendar:read') == 'allowed', 'calendar:read is one of the words of the scope claim'
    plain = 'files:read'
    assert v(sign_token(make_claims(scope='Files:Read')), scope=plain) == 'denied:missing-scope', f"a scope claim of 'Files:Read' does not hold {plain}: case counts for scopes (no lower-casing)"
    assert v(sign_token(make_claims(scope=plain)), scope='Files:Read') == 'denied:missing-scope', f"a required scope of 'Files:Read' is not the word {plain} in the claim: case counts for scopes (no lower-casing)"
    plain = 'files:read'
    assert v(sign_token(make_claims(scope='FILES:READ')), scope=plain) == 'denied:missing-scope', f"a scope claim of 'FILES:READ' does not hold {plain}: case counts for scopes (no lower-casing)"
    assert v(sign_token(make_claims(scope=plain)), scope='FILES:READ') == 'denied:missing-scope', f"a required scope of 'FILES:READ' is not the word {plain} in the claim: case counts for scopes (no lower-casing)"
    plain = 'calendar:read'
    assert v(sign_token(make_claims(scope='Calendar:Read')), scope=plain) == 'denied:missing-scope', f"a scope claim of 'Calendar:Read' does not hold {plain}: case counts for scopes (no lower-casing)"
    assert v(sign_token(make_claims(scope=plain)), scope='Calendar:Read') == 'denied:missing-scope', f"a required scope of 'Calendar:Read' is not the word {plain} in the claim: case counts for scopes (no lower-casing)"
    plain = 'calendar:read'
    assert v(sign_token(make_claims(scope='CALENDAR:READ')), scope=plain) == 'denied:missing-scope', f"a scope claim of 'CALENDAR:READ' does not hold {plain}: case counts for scopes (no lower-casing)"
    assert v(sign_token(make_claims(scope=plain)), scope='CALENDAR:READ') == 'denied:missing-scope', f"a required scope of 'CALENDAR:READ' is not the word {plain} in the claim: case counts for scopes (no lower-casing)"
    assert v(good, scope="files") == "denied:missing-scope", "files is not a word of the claim files:read"
    assert v(good, scope="files:write") == "denied:missing-scope", "files:write is not a word of the claim files:read"
    assert v(sign_token(make_claims(scope="")), scope=None) == "allowed", "an empty scope claim is fine when no scope is required"
    assert v(good, scope="") == "denied:missing-scope", "a required scope of the empty string is still a required scope (only None means none)"
    assert v(sign_token(make_claims(scope="")), scope="files:read") == "denied:missing-scope", "an empty scope claim holds no scope"
    assert v(sign_token(make_claims(scope=' files:read')), scope='') == 'denied:missing-scope', "scope claim ' files:read': str.split() leaves no empty word, so a required scope of '' matches no claim (only None means none)"
    assert v(sign_token(make_claims(scope=' files:read')), scope='files:read') == 'allowed', "scope claim ' files:read': whitespace at either end is not part of a word"
    assert v(sign_token(make_claims(scope='files:read ')), scope='') == 'denied:missing-scope', "scope claim 'files:read ': str.split() leaves no empty word, so a required scope of '' matches no claim (only None means none)"
    assert v(sign_token(make_claims(scope='files:read ')), scope='files:read') == 'allowed', "scope claim 'files:read ': whitespace at either end is not part of a word"
    assert v(sign_token(make_claims(scope='\tfiles:read\n')), scope='') == 'denied:missing-scope', "scope claim '\\tfiles:read\\n': str.split() leaves no empty word, so a required scope of '' matches no claim (only None means none)"
    assert v(sign_token(make_claims(scope='\tfiles:read\n')), scope='files:read') == 'allowed', "scope claim '\\tfiles:read\\n': whitespace at either end is not part of a word"
    both = sign_token(make_claims(scope="calendar:read files:read"))
    assert v(both, scope="calendar:read files:read") == "denied:missing-scope", "a required scope is one word: two words joined by a space are not one word of the claim"
    spaced = sign_token(make_claims(scope="calendar:read  files:read\tfiles:write"))
    assert v(spaced, scope="files:read") == "allowed", "the scope claim is split on any whitespace, runs of it included: a word after two spaces"
    assert v(spaced, scope="files:write") == "allowed", "the scope claim is split on any whitespace, runs of it included: a word after a tab"
    lined = sign_token(make_claims(scope="calendar:read\nfiles:read\r\nfiles:write"))
    assert v(lined, scope="files:read") == "allowed", "the scope claim is split on any whitespace: a word after a newline"
    assert v(lined, scope="files:write") == "allowed", "the scope claim is split on any whitespace: a word after a carriage return and a newline"
    assert v(good, scope=' files:read') == 'denied:missing-scope', "a required scope of ' files:read' is not the word files:read: the scope argument is compared exactly as given, never stripped or trimmed"
    assert v(good, scope='files:read ') == 'denied:missing-scope', "a required scope of 'files:read ' is not the word files:read: the scope argument is compared exactly as given, never stripped or trimmed"
    assert v(good, scope='\tfiles:read') == 'denied:missing-scope', "a required scope of '\\tfiles:read' is not the word files:read: the scope argument is compared exactly as given, never stripped or trimmed"
    assert v(good, scope='files:read\n') == 'denied:missing-scope', "a required scope of 'files:read\\n' is not the word files:read: the scope argument is compared exactly as given, never stripped or trimmed"
    assert v(good, scope='\nfiles:read\n') == 'denied:missing-scope', "a required scope of '\\nfiles:read\\n' is not the word files:read: the scope argument is compared exactly as given, never stripped or trimmed"
    exotic = sign_token(make_claims(scope="calendar:read\x0bfiles:read\x0cfiles:write\xa0mail:send"))
    assert v(exotic, scope='calendar:read') == 'allowed', 'calendar:read: str.split() separates words at any whitespace character: a vertical tab, a form feed and a non-breaking space included (a split on spaces, tabs and newlines alone misses them)'
    assert v(exotic, scope='files:read') == 'allowed', 'files:read: str.split() separates words at any whitespace character: a vertical tab, a form feed and a non-breaking space included (a split on spaces, tabs and newlines alone misses them)'
    assert v(exotic, scope='files:write') == 'allowed', 'files:write: str.split() separates words at any whitespace character: a vertical tab, a form feed and a non-breaking space included (a split on spaces, tabs and newlines alone misses them)'
    assert v(exotic, scope='mail:send') == 'allowed', 'mail:send: str.split() separates words at any whitespace character: a vertical tab, a form feed and a non-breaking space included (a split on spaces, tabs and newlines alone misses them)'
    assert v(sign_token(make_claims(extra="anything", note=[1, 2]))) == "allowed", "claims the verifier does not know are ignored, not refused"
    assert v(sign_token(without(make_claims(), "iat"))) == "allowed", "the verifier does not look at iat: a token without one is good if it passes the eight checks"
    assert v(sign_token(make_claims(iss=None, iat="yesterday"))) == "allowed", "iss and iat are not looked at at all, whatever type they hold: only the eight checks decide"
    bare_head, bare_body = _b64(_json({"alg": "HS256"})), _b64(_json(without(make_claims(), "iss", "iat")))
    assert v(f"{bare_head}.{bare_body}.{mac(bare_head + '.' + bare_body, key)}") == "allowed", (
        "a header with only alg (no typ) and claims with no iss and no iat are good: the header chooses nothing and iss, iat and typ are optional")
    assert v(sign_token(make_claims(sub="", jti=""))) == "allowed", "an empty string is a string: only the type of sub, aud, jti and scope is checked, not that they are non-empty"
    assert v(sign_token(make_claims(nbf=-1))) == "allowed", "an nbf below zero is still an int, and it is already satisfied: only the type of nbf is checked, and now < nbf decides"
    assert v(sign_token(make_claims(exp=0))) == 'denied:expired', "an exp of 0 is an int and long past: 'expired', not 'bad-claims' (only the type of exp is checked, of any sign)"
    assert v(sign_token(make_claims(exp=-1))) == 'denied:expired', "an exp of -1 is an int and long past: 'expired', not 'bad-claims' (only the type of exp is checked, of any sign)"
    assert v(sign_token(make_claims(exp=-10 ** 9))) == 'denied:expired', "an exp of -1000000000 is an int and long past: 'expired', not 'bad-claims' (only the type of exp is checked, of any sign)"
    kid_head = _b64(_json({"alg": "HS256", "kid": "k1"}))
    assert v(f"{kid_head}.{body}.{mac(kid_head + '.' + body, key)}") == "allowed", "a header key the verifier does not know (kid) is ignored: only alg is read from the header"
    typ_head = _b64(_json({'alg': 'HS256', 'typ': 'at+jwt'}))
    assert v(f"{typ_head}.{body}.{mac(typ_head + '.' + body, key)}") == 'allowed', "a typ of 'at+jwt' is ignored: only alg is read from the header"
    typ_head = _b64(_json({'alg': 'HS256', 'typ': ''}))
    assert v(f"{typ_head}.{body}.{mac(typ_head + '.' + body, key)}") == 'allowed', "a typ of '' is ignored: only alg is read from the header"
    typ_head = _b64(_json({'alg': 'HS256', 'typ': 5}))
    assert v(f"{typ_head}.{body}.{mac(typ_head + '.' + body, key)}") == 'allowed', 'a typ of 5 is ignored: only alg is read from the header'
    typ_head = _b64(_json({'alg': 'HS256', 'typ': None}))
    assert v(f"{typ_head}.{body}.{mac(typ_head + '.' + body, key)}") == 'allowed', 'a typ of None is ignored: only alg is read from the header'
    url_claims = make_claims(jti="j-1?")
    url_body = _b64(_json(url_claims))
    if "-" not in url_body and "_" not in url_body:         # fixture check, not a case
        raise RuntimeError("the fixture payload does not use - or _")
    assert v(f"{head}.{url_body}.{mac(head + '.' + url_body, key)}") == "allowed", (
        "the parts are base64url, with - and _: decode them with the given _unb64, never base64.b64decode (its alphabet is + and /)")
    url_head = _b64(_json({"alg": "HS256", "typ": "JWT", "kid": "k1?>?"}))
    if "-" not in url_head and "_" not in url_head:         # fixture check, not a case
        raise RuntimeError("the fixture header does not use - or _")
    assert v(f"{url_head}.{body}.{mac(url_head + '.' + body, key)}") == "allowed", (
        "the header is base64url too, with - and _: decode it with the given _unb64, never base64.b64decode")
    print("exercise 1 looks right")


if __name__ == "__main__":
    _try("exercise 1", _check_verify)

## 4. Exercise 2: issue the agent a narrower token, on the user's behalf

RFC 8693 (OAuth 2.0 Token Exchange) lets a client trade one token for another. Its `act` claim says that delegation
has happened and who the acting party is (section 13 gives the sources and the dates they were read). Here the
*subject* is the user's login token and the *actor* is the agent: the new token keeps the user in `sub`, puts the
agent in `act`, and carries **less** than the user could do, for one service, for a few minutes.

**`exchange(subject_token, agent_id, scopes, audience, now, jti, key=SIGNING_KEY)`** returns a signed token string
(made with `sign_token(claims, key)`), or raises `Denied(code)` for the first problem, in this order:

1. the subject token must pass `verify_token(subject_token, key, now, STS)`; its `Denied` is raised as it is;
2. `"unknown-agent"`: `agent_id` is not a key of `AGENTS`;
3. `"audience-not-allowed"`: `audience` is not in that agent's `audiences`, compared exactly (`Files-API` and `files-api ` are not `files-api`), even when the requested scopes would also be an escalation;
4. `"no-scope"`: `scopes` is empty (a list or a tuple of no items; a list that holds the empty string is not empty: it asks for a scope nobody holds);
5. `"scope-escalation"`: any requested scope is missing from the scopes the subject token holds **or** missing from the
   agent's `scopes`. The subject token's scopes are the whole words of its `scope` claim, as in exercise 1, split on any whitespace
   (so `calendar:readonly` gives no `calendar:read`, and a tab or a newline separates two scopes). Case counts for scopes: `FILES:WRITE` is not
   `files:write`, whichever side holds it. A requested scope is one string, compared as it is
   (`"files:read files:write"` is one scope that nobody holds). The claim is the user's bound, not the table
   `USERS`: a claim may hold more than `USERS` lists for that user. The scopes of a request are compared exactly as given, never stripped
   (`" files:read"` and `"files:read\n"` are not `files:read`), and the audience does not limit which held scopes may be asked for:
   `files:read` is allowed for `calendar-api` when the subject token and the agent both hold it. The request is refused as a whole; it is never trimmed to what
   would have been allowed.

The token it signs has exactly these claims: `iss` = `STS` (always, whatever `iss` the subject token holds: nothing looks at it),
`sub` = the subject token's `sub` (unchanged, whoever it names), `act` = `{"sub": agent_id}` (exactly that, even when the subject token
already carries an `act` claim of its own: nothing is copied from it and no chain of actors is built),
`aud` = `audience`, `scope` = the requested scopes without duplicates, sorted, joined by single spaces, `iat` = `now`,
`exp` = the smaller of `now + TTL` and the subject token's `exp` (an agent's token never outlives the login it came
from), and `jti` = the `jti` argument. Never change `scopes`.

<details><summary>💡 Hint 1 — what to think about</summary>

The user's authority is an upper bound and the agent's registration is a second upper bound: the token may hold only
what both allow. Which check should come first, the one that needs the subject token's claims, or the one that does not?
What would trimming silently hide from the person who asked?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Verify the subject token first and keep its claims. Look the agent up in `AGENTS`. Turn the requested scopes into a
set and compare it with the intersection of two sets: the subject's scope words and the agent's scopes. Build the
claims dict with the eight names above and sign it with the key you were given.

</details>

In [ ]:
def exchange(subject_token: str, agent_id: str, scopes, audience: str, now: int, jti: str, key: bytes = SIGNING_KEY) -> str:
    """A delegated token for `agent_id` acting for the subject's user, as described above.

    Example:
        >>> login = sign_token({"sub": "alice", "aud": STS, "scope": "files:read files:write", "exp": T0 + 3600, "jti": "l"})
        >>> claims = peek_claims(exchange(login, "agent-scribe", ["files:read"], "files-api", T0, "t-1"))
        >>> claims["sub"], claims["act"], claims["scope"], claims["exp"] - T0
        ('alice', {'sub': 'agent-scribe'}, 'files:read', 300)

    Returns:
        The signed token; Denied(code) when the exchange is refused.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def login_token(user: str = "alice", scope=None, exp=None, **over) -> str:
    """A login token for `user` as the token service would sign it (scopes default to the user's whole grant)."""
    claims = {"iss": STS, "sub": user, "aud": STS, "scope": " ".join(sorted(USERS[user] if scope is None else scope)),
              "iat": T0, "exp": T0 + SUBJECT_TTL if exp is None else exp, "jti": f"login-{user}-0001"}
    claims.update(over)
    return sign_token(claims)


def _check_exchange() -> None:
    now = T0 + 10

    def took(why, *args):
        """The token exchange(*args) returns; a refusal is reported with the case's own message, which names the likely mistake."""
        try:
            return exchange(*args)
        except Denied as exc:
            raise AssertionError(f"exchange refused this request with Denied('{exc.code}') and should have allowed it: {why}") from None

    login = login_token()
    want = {"iss": STS, "sub": "alice", "act": {"sub": "agent-scribe"}, "aud": "files-api", "scope": "files:read",
            "iat": now, "exp": now + TTL, "jti": "j-9"}
    token = took("a first plain request", login, "agent-scribe", ["files:read"], "files-api", now, "j-9")
    assert isinstance(token, str), "exchange returns the signed token text, a str (the value of sign_token)"
    out = peek_claims(token)
    assert out.get("iss") == STS, "iss is STS"
    assert out.get("sub") == "alice", "sub is copied from the SUBJECT token (the login token's user)"
    assert out.get("act") == {"sub": "agent-scribe"}, "act is exactly {'sub': agent_id}: it names the agent"
    assert out.get("aud") == "files-api", "aud is the audience argument"
    assert out.get("scope") == "files:read", "scope is the requested scopes as one text"
    assert out.get("iat") == now, "iat is now"
    assert out.get("exp") == now + TTL, "exp is now + TTL here (the smaller of now + TTL and the subject token's exp)"
    assert out.get("jti") == "j-9", "jti is the jti argument"
    assert sorted(out) == sorted(want), "exactly iss, sub, act, aud, scope, iat, exp and jti, and nothing else"
    assert peek_claims(took("iss is STS whatever iss the login token holds: nothing looks at the login token's iss", login_token("alice", iss="idp"), "agent-scribe", ["files:read"], "files-api", now, "j-9"))["iss"] == STS, (
        "iss is STS whatever iss the login token holds: do not copy it from the subject token (nothing looks at the login token's iss)")
    noisy = login_token("alice", nbf=T0, extra="kept", note=[1, 2])
    assert sorted(peek_claims(took("a login token with an nbf and claims the exchange does not know is still a good login token", noisy, "agent-scribe", ["files:read"], "files-api", now, "j-9"))) == sorted(want), (
        "the new token holds exactly iss, sub, act, aud, scope, iat, exp and jti: nothing else is copied from the subject token (not its nbf, not a claim you do not know)")
    other = b"another key"
    signed_other = took("the key argument verifies the login token and signs the new one", sign_token(peek_claims(login), other), "agent-scribe", ["files:read"], "files-api", now, "j-9", other)
    assert signed_other.split(".")[2] == mac(".".join(signed_other.split(".")[:2]), other), "the key argument signs the new token (it also verifies the login token: that is why this login token is signed with it)"
    got = peek_claims(took("a request that names a scope twice is still a good request", login, "agent-scribe", ["files:write", "files:read", "files:read"], "files-api", now, "j-2"))["scope"]
    assert got == "files:read files:write", f"got {got!r}: no duplicates, sorted, one space between scopes"
    short = login_token(exp=now + 100)
    assert peek_claims(took("a login token that ends soon is still a good login token", short, "agent-scribe", ["files:read"], "files-api", now, "j-3"))["exp"] == now + 100, (
        "the agent's token must not outlive the login token it was exchanged from: exp is the smaller of now + TTL and the subject's exp")
    assert peek_claims(took(f'a login token that ends at now + {now + TTL - 1 - now} is still good', login_token(exp=now + TTL - 1), 'agent-scribe', ['files:read'], 'files-api', now, 'j-3'))['exp'] == now + TTL - 1, f"the login token ends at now + {now + TTL - 1 - now}: so does the agent's token (the smaller of now + TTL and the subject's exp)"
    assert peek_claims(took(f'a login token that ends at now + {now + TTL - now} is still good', login_token(exp=now + TTL), 'agent-scribe', ['files:read'], 'files-api', now, 'j-3'))['exp'] == now + TTL, f"the login token ends at now + {now + TTL - now}: so does the agent's token (the smaller of now + TTL and the subject's exp)"
    assert peek_claims(took("a login token that outlives now + TTL is still good", login_token(exp=now + TTL + 1), "agent-scribe", ["files:read"], "files-api", now, "j-3"))["exp"] == now + TTL, "otherwise now + TTL"
    asked = ["files:write", "files:read"]
    took("a request of two held scopes is allowed", login, "agent-scribe", asked, "files-api", now, "j-4")
    assert asked == ["files:write", "files:read"], "never change the scopes argument (sorting a copy is fine)"
    assert _code(exchange, login, 'agent-rogue', ['files:read'], 'files-api', now, 'j-5') == 'denied:unknown-agent', "{'agent_id': 'agent-rogue'}: expected Denied('unknown-agent'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, '', ['files:read'], 'files-api', now, 'j-5') == 'denied:unknown-agent', "{'agent_id': ''}: expected Denied('unknown-agent'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['files:read'], 'mail-api', now, 'j-5') == 'denied:audience-not-allowed', "{'audience': 'mail-api'}: expected Denied('audience-not-allowed'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['mail:send'], 'mail-api', now, 'j-5') == 'denied:audience-not-allowed', "{'audience': 'mail-api', 'scopes': ['mail:send']}: expected Denied('audience-not-allowed'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['files:read', 'calendar:write'], 'mail-api', now, 'j-5') == 'denied:audience-not-allowed', "{'audience': 'mail-api', 'scopes': ['files:read', 'calendar:write']}: expected Denied('audience-not-allowed'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['files:read'], STS, now, 'j-5') == 'denied:audience-not-allowed', "{'audience': 'sts'}: expected Denied('audience-not-allowed'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['files:read'], 'Files-API', now, 'j-5') == 'denied:audience-not-allowed', "{'audience': 'Files-API'}: expected Denied('audience-not-allowed'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['files:read'], 'files-api ', now, 'j-5') == 'denied:audience-not-allowed', "{'audience': 'files-api '}: expected Denied('audience-not-allowed'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['files:read'], ' files-api', now, 'j-5') == 'denied:audience-not-allowed', "{'audience': ' files-api'}: expected Denied('audience-not-allowed'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['calendar:write'], 'FILES-API', now, 'j-5') == 'denied:audience-not-allowed', "{'audience': 'FILES-API', 'scopes': ['calendar:write']}: expected Denied('audience-not-allowed'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['files:read'], 'Calendar-API', now, 'j-5') == 'denied:audience-not-allowed', "{'audience': 'Calendar-API'}: expected Denied('audience-not-allowed'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['files:read'], 'calendar-api ', now, 'j-5') == 'denied:audience-not-allowed', "{'audience': 'calendar-api '}: expected Denied('audience-not-allowed'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['files:read'], ' calendar-api', now, 'j-5') == 'denied:audience-not-allowed', "{'audience': ' calendar-api'}: expected Denied('audience-not-allowed'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['files:write'], 'CALENDAR-API', now, 'j-5') == 'denied:audience-not-allowed', "{'audience': 'CALENDAR-API', 'scopes': ['files:write']}: expected Denied('audience-not-allowed'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', [], 'files-api', now, 'j-5') == 'denied:no-scope', "{'scopes': []}: expected Denied('no-scope'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', (), 'files-api', now, 'j-5') == 'denied:no-scope', "{'scopes': ()}: expected Denied('no-scope'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['calendar:write'], 'files-api', now, 'j-5') == 'denied:scope-escalation', "{'scopes': ['calendar:write']}: expected Denied('scope-escalation'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['mail:send'], 'files-api', now, 'j-5') == 'denied:scope-escalation', "{'scopes': ['mail:send'], 'audience': 'files-api'}: expected Denied('scope-escalation'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['files:read', 'calendar:write'], 'files-api', now, 'j-5') == 'denied:scope-escalation', "{'scopes': ['files:read', 'calendar:write']}: expected Denied('scope-escalation'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['files'], 'files-api', now, 'j-5') == 'denied:scope-escalation', "{'scopes': ['files']}: expected Denied('scope-escalation'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['unknown:scope'], 'files-api', now, 'j-5') == 'denied:scope-escalation', "{'scopes': ['unknown:scope']}: expected Denied('scope-escalation'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', [''], 'files-api', now, 'j-5') == 'denied:scope-escalation', "{'scopes': ['']}: expected Denied('scope-escalation'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['files:read', ''], 'files-api', now, 'j-5') == 'denied:scope-escalation', "{'scopes': ['files:read', '']}: expected Denied('scope-escalation'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['files:read files:write'], 'files-api', now, 'j-5') == 'denied:scope-escalation', "{'scopes': ['files:read files:write']}: expected Denied('scope-escalation'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['Files:Read'], 'files-api', now, 'j-5') == 'denied:scope-escalation', "{'scopes': ['Files:Read']}: expected Denied('scope-escalation'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['FILES:READ'], 'files-api', now, 'j-5') == 'denied:scope-escalation', "{'scopes': ['FILES:READ']}: expected Denied('scope-escalation'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['Calendar:Read'], 'files-api', now, 'j-5') == 'denied:scope-escalation', "{'scopes': ['Calendar:Read']}: expected Denied('scope-escalation'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    assert _code(exchange, login, 'agent-scribe', ['CALENDAR:READ'], 'files-api', now, 'j-5') == 'denied:scope-escalation', "{'scopes': ['CALENDAR:READ']}: expected Denied('scope-escalation'). A request with one bad scope among good ones is refused whole, not trimmed, and the audience is tested before the scopes"
    bob = login_token("bob")
    assert _code(exchange, bob, "agent-scribe", ["files:write"], "files-api", now, "j-6") == "denied:scope-escalation", (
        "the agent may hold files:write but bob's login does not: the token holds only what BOTH allow")
    assert _code(exchange, bob, "agent-scribe", ["calendar:read"], "calendar-api", now, "j-6") == "allowed", "bob's login holds calendar:read and the agent is registered for it: the request is allowed (do not refuse a scope that both bounds allow)"
    near = login_token("alice", scope=["calendar:readonly"])
    assert _code(exchange, near, "agent-scribe", ["calendar:read"], "calendar-api", now, "j-6") == "denied:scope-escalation", (
        "the login token's scope claim is read by whole words: 'calendar:readonly' does not give 'calendar:read' (no substring or prefix test)")
    both = login_token("alice", scope=["files:read", "files:write"])
    assert _code(exchange, both, "agent-scribe", ["files:write"], "files-api", now, "j-6") == "allowed", "a login token that holds the scope allows the request"
    assert _code(exchange, login, 'agent-scribe', [' files:read'], 'files-api', now, 'j-6') == 'denied:scope-escalation', "a requested scope of ' files:read' is not files:read: requested scopes are compared exactly as given, never stripped or trimmed"
    assert _code(exchange, login, 'agent-scribe', ['files:read', ' files:read'], 'files-api', now, 'j-6') == 'denied:scope-escalation', "a request that holds ' files:read' among good scopes is refused whole: compared exactly as given"
    assert _code(exchange, login, 'agent-scribe', ['files:read '], 'files-api', now, 'j-6') == 'denied:scope-escalation', "a requested scope of 'files:read ' is not files:read: requested scopes are compared exactly as given, never stripped or trimmed"
    assert _code(exchange, login, 'agent-scribe', ['files:read', 'files:read '], 'files-api', now, 'j-6') == 'denied:scope-escalation', "a request that holds 'files:read ' among good scopes is refused whole: compared exactly as given"
    assert _code(exchange, login, 'agent-scribe', ['\tfiles:read'], 'files-api', now, 'j-6') == 'denied:scope-escalation', "a requested scope of '\\tfiles:read' is not files:read: requested scopes are compared exactly as given, never stripped or trimmed"
    assert _code(exchange, login, 'agent-scribe', ['files:read', '\tfiles:read'], 'files-api', now, 'j-6') == 'denied:scope-escalation', "a request that holds '\\tfiles:read' among good scopes is refused whole: compared exactly as given"
    assert _code(exchange, login, 'agent-scribe', ['files:read\n'], 'files-api', now, 'j-6') == 'denied:scope-escalation', "a requested scope of 'files:read\\n' is not files:read: requested scopes are compared exactly as given, never stripped or trimmed"
    assert _code(exchange, login, 'agent-scribe', ['files:read', 'files:read\n'], 'files-api', now, 'j-6') == 'denied:scope-escalation', "a request that holds 'files:read\\n' among good scopes is refused whole: compared exactly as given"
    assert peek_claims(took('files:read for calendar-api: the audience does not limit which held scopes may be asked for', login, 'agent-scribe', ['files:read'], 'calendar-api', now, 'j-7'))['scope'] == 'files:read', 'files:read for calendar-api: the audience does not limit which held scopes may be asked for, when the subject token and the agent both hold them'
    assert peek_claims(took('calendar:read for files-api: the audience does not limit which held scopes may be asked for', login, 'agent-scribe', ['calendar:read'], 'files-api', now, 'j-7'))['scope'] == 'calendar:read', 'calendar:read for files-api: the audience does not limit which held scopes may be asked for, when the subject token and the agent both hold them'
    exotic = login_token("alice", scope=["calendar:read\x0bfiles:read\x0cfiles:write\xa0mail:send"])
    assert _code(exchange, exotic, 'agent-scribe', ['calendar:read'], 'calendar-api', now, 'j-6') == 'allowed', "the subject token's scope claim is split like str.split(): a vertical tab, a form feed and a non-breaking space separate scopes too (calendar:read is held)"
    assert _code(exchange, exotic, 'agent-scribe', ['files:read'], 'files-api', now, 'j-6') == 'allowed', "the subject token's scope claim is split like str.split(): a vertical tab, a form feed and a non-breaking space separate scopes too (files:read is held)"
    assert _code(exchange, exotic, 'agent-scribe', ['files:write'], 'files-api', now, 'j-6') == 'allowed', "the subject token's scope claim is split like str.split(): a vertical tab, a form feed and a non-breaking space separate scopes too (files:write is held)"
    assert _code(exchange, exotic, "agent-scribe", ["mail:send"], "files-api", now, "j-6") == "denied:scope-escalation", "the agent does not hold mail:send, whatever the claim holds"
    tabbed = login_token("alice", scope=["files:read\tfiles:write"])
    assert _code(exchange, tabbed, "agent-scribe", ["files:write"], "files-api", now, "j-6") == "allowed", (
        "the subject token's scope claim is split on any whitespace, a tab included: a claim of files:read TAB files:write holds files:write")
    wide = login_token("bob", scope=["calendar:read", "files:write"])
    assert _code(exchange, wide, "agent-scribe", ["files:write"], "files-api", now, "j-6") == "allowed", (
        "the claim of the login token you were given is the user's bound, not the table USERS: this claim holds more than USERS lists for bob")
    lined = login_token("alice", scope=["files:read\nfiles:write"])
    assert _code(exchange, lined, "agent-scribe", ["files:write"], "files-api", now, "j-6") == "allowed", (
        "the subject token's scope claim is split on any whitespace, a newline included: a claim of files:read NEWLINE files:write holds files:write")
    assert peek_claims(took("scopes may be a tuple as well as a list", login, "agent-scribe", ("files:write", "files:read"), "files-api", now, "j-6"))["scope"] == "files:read files:write", (
        "scopes may be a tuple as well as a list: same sorted, de-duplicated text")
    assert _code(exchange, login_token('alice', scope=['FILES:WRITE']), 'agent-scribe', ['files:write'], 'files-api', now, 'j-6') == 'denied:scope-escalation', "a login token whose scope claim is 'FILES:WRITE' does not hold 'files:write': case counts for scopes (no lower-casing)"
    assert _code(exchange, login_token('alice', scope=['Files:Read']), 'agent-scribe', ['files:read'], 'files-api', now, 'j-6') == 'denied:scope-escalation', "a login token whose scope claim is 'Files:Read' does not hold 'files:read': case counts for scopes (no lower-casing)"
    assert _code(exchange, login_token('alice', scope=['CALENDAR:READ']), 'agent-scribe', ['calendar:read'], 'calendar-api', now, 'j-6') == 'denied:scope-escalation', "a login token whose scope claim is 'CALENDAR:READ' does not hold 'calendar:read': case counts for scopes (no lower-casing)"
    assert _code(exchange, login_token('alice', scope=['Calendar:Read']), 'agent-scribe', ['calendar:read'], 'calendar-api', now, 'j-6') == 'denied:scope-escalation', "a login token whose scope claim is 'Calendar:Read' does not hold 'calendar:read': case counts for scopes (no lower-casing)"
    chained = login_token("alice", act={"sub": "agent-old", "act": {"sub": "agent-older"}})
    assert peek_claims(took("a login token that already carries an act claim is a good login token", chained, "agent-scribe", ["files:read"], "files-api", now, "j-9"))["act"] == {"sub": "agent-scribe"}, (
        "a login token that already carries an act claim passes none of it on: the new token's act is exactly {'sub': agent_id}, with no chain and no copy")
    assert peek_claims(took("a login token that already names an actor is a good login token", login_token("alice", act={"sub": "agent-old"}), "agent-scribe", ["files:read"], "files-api", now, "j-9"))["act"] == {"sub": "agent-scribe"}, (
        "a login token that already names an actor (act) passes it on to no one: the new token's act is exactly {'sub': agent_id}")
    narrow = login_token("alice", scope=["files:read"])
    assert _code(exchange, narrow, "agent-scribe", ["calendar:read"], "calendar-api", now, "j-6") == "denied:scope-escalation", (
        "the scope claim of the login token you were given is the user's bound here, not a table of what the user could hold")
    assert _code(exchange, login, "agent-rogue", ["files:read"], "mail-api", now, "j") == "denied:unknown-agent", "unknown agent comes before audience"
    assert _code(exchange, login, "agent-scribe", [], "mail-api", now, "j") == "denied:audience-not-allowed", "audience comes before an empty scope list"
    assert _code(exchange, login, "agent-scribe", [], "files-api", now, "j") == "denied:no-scope", "an empty list of scopes for an allowed audience is 'no-scope' (an empty list is a subset of everything: test for it, do not let it through)"
    assert _code(exchange, login_token(exp=now), "agent-rogue", ["files:read"], "files-api", now, "j") == "denied:expired", (
        "the subject token is verified first, and its own reason is raised as it is")
    assert _code(exchange, sign_token(want), 'agent-scribe', ['files:read'], 'files-api', now, 'j') == 'denied:wrong-audience', "a subject token that fails verification raises verify_token's own Denied('wrong-audience')"
    assert _code(exchange, login[:-2] + 'AA', 'agent-scribe', ['files:read'], 'files-api', now, 'j') == 'denied:bad-signature', "a subject token that fails verification raises verify_token's own Denied('bad-signature')"
    assert _code(exchange, 'nonsense', 'agent-scribe', ['files:read'], 'files-api', now, 'j') == 'denied:malformed', "a subject token that fails verification raises verify_token's own Denied('malformed')"
    assert peek_claims(took("a login token of bob is a good login token", login_token("bob"), "agent-scribe", ["calendar:read"], "calendar-api", now, "j-8"))["sub"] == "bob", (
        "sub is copied from the subject token, whoever the user is")
    assert peek_claims(took("a login token of a user that USERS does not list is still a good login token", login_token("alice", sub=" Carol Ng "), "agent-scribe", ["files:read"], "files-api", now, "j-8"))["sub"] == " Carol Ng ", (
        "sub is copied unchanged: not looked up in USERS, not lower-cased, not trimmed")
    print("exercise 2 looks right")


if __name__ == "__main__":
    _try("exercise 2", _check_exchange, needs=("exercise 1",))

In [ ]:
def show_agent_token() -> None:
    token = exchange(login_token(), "agent-scribe", ["files:read"], "files-api", T0, "agent-token-0001")
    print("the agent's token for files-api, claims:")
    for name, value in peek_claims(token).items():
        print(f"  {name:6} {value}")
    print(f"sub is {peek_claims(token)['sub']!r} (whose authority), act names {peek_claims(token)['act']['sub']!r} (who is using it); "
          f"it holds {len(peek_claims(token)['scope'].split())} of the {len(USERS['alice'])} scopes alice holds")


if __name__ == "__main__":
    _try("agent token demo", show_agent_token, needs=("exercise 1", "exercise 2"))

## 5. Exercise 3: the service that accepts or refuses a call

`ResourceServer` is what a service runs in front of its operations. You already have the verifier; this exercise
adds what a verifier cannot know on its own: that a token was **revoked**, that a single-use token was **already spent**,
and who to put in the **audit log**. State lives in three attributes the constructor sets (given):
`self.revoked` (a set of `jti` values, shared with other servers when one is passed in), `self.used` (a dict, `jti`
to the token's `exp`, for single-use tokens already spent) and `self.audit` (a list).

**`handle(token, op, now)`** returns the verified claims or raises `Denied(code)`. In this order:

0. first of all, always, even when the call is then refused: delete from `self.used` every entry whose `exp <= now`
   (a replayed token past its `exp` is refused as expired anyway, so remembering it only grows memory);
1. `"unknown-op"`: `op` is not in `OPS`, or the service that owns it (`OPS[op][0]`) is not `self.audience`; `op` is compared exactly as given,
   never stripped or trimmed (`" read_file"` and `"read_file\n"` are unknown);
2. verify with `verify_token(token, self.key, now, self.audience, OPS[op][1])`; its `Denied` is raised as it is. The scope test is
   the verifier's, so it is whole-word and comes before revocation (`files:writeonly` does not give `files:write`, and a revoked
   token that lacks the scope is `"missing-scope"`). Read nothing from a token before this step has passed: a token that does not
   even decode is `"malformed"`, whatever the operation;
3. `"revoked"`: the token's `jti` is in `self.revoked`;
4. for an `op` in `SINGLE_USE` only (`delete_file` and `send_mail`; every other operation, `read_events`, `add_event` and `read_file` included,
   may be called again and again with the same token and never records it): `"replayed"` when the `jti` is already in `self.used`; otherwise record
   `self.used[jti] = exp`. A call refused at any step above must **not** use up its token;
5. on success append the tuple `(op, sub, actor)` to `self.audit` (`actor` is the `sub` inside the claim `act`, or
   `None` when there is no `act` claim or it holds no `sub`: an `act` that is not a dict, such as a string or a list, holds
   no `sub`; a `sub` that is there is copied as it is, so an int or an empty string is recorded unchanged), and return the
   claims. A refused call writes nothing to `self.audit`, whatever the reason it was refused for (`"unknown-op"`, every reason of the verifier,
   `"revoked"` and `"replayed"` included), and a refused call changes `self.used` for no reason either (apart from the clean-up in step 0).

<details><summary>💡 Hint 1 — what to think about</summary>

Which of these facts does the signature already guarantee, and which does only the server remember? When is it safe
to record that a token has been spent, and what goes wrong if you record it earlier than that? Reads are not single
use: should they enter the spent set at all?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Start with the clean-up of `self.used`, then the operation check, then call the verifier from exercise 1. Test
revocation on the claims it returns. Only then, for a single-use operation, test and record the `jti`. Append to the
audit last, so that nothing is written for a call that was refused.

</details>

In [ ]:
class ResourceServer:
    """The checks one service runs before it carries out an operation."""

    def __init__(self, audience: str, key: bytes = SIGNING_KEY, revoked=None):
        self.audience = audience
        self.key = key
        self.revoked = revoked if revoked is not None else set()    # jti values; shared by reference when passed in
        self.used = {}                                              # jti -> exp of single-use tokens already spent
        self.audit = []                                             # (op, sub, actor) of every allowed call

    def handle(self, token: str, op: str, now: int) -> dict:
        """Accept or refuse one call, as described above.

        Example:
            >>> server = ResourceServer("files-api")
            >>> token = sign_token(make_claims(scope="files:write", jti="d-1"))
            >>> sorted(server.handle(token, "delete_file", T0 + 1))[:2], server.audit
            (['act', 'aud'], [('delete_file', 'alice', 'agent-scribe')])
            >>> _code(server.handle, token, "delete_file", T0 + 2)
            'denied:replayed'

        Returns:
            The verified claims; Denied(code) when the call is refused.
        """
        # YOUR CODE HERE
        raise NotImplementedError


def _check_server() -> None:
    now = T0 + 10
    server = ResourceServer("files-api")

    def go(token, op, at=now, srv=None):
        return _code((srv or server).handle, token, op, at)

    read = sign_token(make_claims(jti="r-1"))
    calendar = ResourceServer("calendar-api")
    server = ResourceServer("files-api")
    assert server.handle(read, "read_file", now) == make_claims(jti="r-1"), "an allowed call returns the verified claims"
    assert len(server.audit) == 1, f"got {server.audit}: an allowed call appends exactly one line to the audit"
    assert type(server.audit[0]) is tuple, f"got {server.audit[0]!r}: the audit line is a tuple (op, sub, actor), not a list"
    assert len(server.audit[0]) == 3, f"got {server.audit[0]!r}: the audit line holds exactly op, sub and actor, nothing more"
    assert server.audit[0][0] == "read_file", f"got {server.audit[0]!r}: the first field of the audit line is the operation"
    assert server.audit[0][1] == "alice", f"got {server.audit[0]!r}: the second field of the audit line is the token's sub"
    assert server.audit[0][2] == "agent-scribe", f"got {server.audit[0]!r}: the third field of the audit line is the actor, the sub inside the act claim"
    server.handle(sign_token(without(make_claims(jti="r-2"), "act")), "read_file", now)
    assert server.audit[-1][2] is None, f"got {server.audit[-1]}: no act claim means the actor is None"
    server.handle(sign_token(make_claims(jti="r-3", act={})), "read_file", now)
    assert server.audit[-1][2] is None, "an act claim that holds no sub gives actor None (no KeyError)"
    server.handle(sign_token(make_claims(jti="r-4", act={"note": "x"})), "read_file", now)
    assert server.audit[-1][2] is None, "a non-empty act claim that holds no sub gives actor None too (an empty dict is falsy, this one is not)"
    server.handle(sign_token(make_claims(jti='r-50', act='agent-scribe')), 'read_file', now)
    assert server.audit[-1][2] is None, f"got {server.audit[-1]}: an act claim that is not a dict ('agent-scribe') holds no sub: the actor is None, and nothing may crash"
    server.handle(sign_token(make_claims(jti='r-51', act=['agent-scribe'])), 'read_file', now)
    assert server.audit[-1][2] is None, f"got {server.audit[-1]}: an act claim that is not a dict (['agent-scribe']) holds no sub: the actor is None, and nothing may crash"
    server.handle(sign_token(make_claims(jti='r-52', act='subagent')), 'read_file', now)
    assert server.audit[-1][2] is None, f"got {server.audit[-1]}: an act claim that is not a dict ('subagent') holds no sub: the actor is None, and nothing may crash"
    server.handle(sign_token(make_claims(jti='r-53', act=['sub'])), 'read_file', now)
    assert server.audit[-1][2] is None, f"got {server.audit[-1]}: an act claim that is not a dict (['sub']) holds no sub: the actor is None, and nothing may crash"
    server.handle(sign_token(make_claims(jti='r-60', act={'sub': 5})), 'read_file', now)
    assert server.audit[-1][2] == 5, f'got {server.audit[-1]}: a sub inside act is copied as it is (5 here), not turned into None or a string'
    assert type(server.audit[-1][2]) is int, f"got {server.audit[-1]}: a sub of 5 inside act is copied as it is, an int stays an int (not 5.0, not '5')"
    server.handle(sign_token(make_claims(jti='r-61', act={'sub': ''})), 'read_file', now)
    assert server.audit[-1][2] == '', f"got {server.audit[-1]}: a sub inside act is copied as it is ('' here), not turned into None or a string"
    server = ResourceServer("files-api")
    assert go(sign_token(make_claims()), "nope") == "denied:unknown-op", "an operation that does not exist is 'unknown-op'"
    assert go(sign_token(make_claims()), "send_mail") == "denied:unknown-op", "an operation another service owns is 'unknown-op' here too"
    assert go(sign_token(make_claims()), "READ_FILE") == "denied:unknown-op", "operation names are compared exactly, case included"
    assert go(sign_token(make_claims()), "") == "denied:unknown-op", "the empty string is not an operation"
    assert go(sign_token(make_claims()), ' read_file') == 'denied:unknown-op', "' read_file': operation names are compared exactly, never stripped or trimmed"
    assert go(sign_token(make_claims()), 'read_file ') == 'denied:unknown-op', "'read_file ': operation names are compared exactly, never stripped or trimmed"
    assert go(sign_token(make_claims()), 'read_file\n') == 'denied:unknown-op', "'read_file\\n': operation names are compared exactly, never stripped or trimmed"
    assert go(sign_token(make_claims()), '\tread_file') == 'denied:unknown-op', "'\\tread_file': operation names are compared exactly, never stripped or trimmed"
    assert go(sign_token(make_claims(), alg="none"), "nope") == "denied:unknown-op", "unknown-op comes before the token is looked at"
    assert go('junk', 'read_file') == 'denied:malformed', "'junk' for read_file: a token that does not decode gets the verifier's own reason: do not read its claims (not even its jti) before verify_token has passed"
    assert go('junk', 'delete_file') == 'denied:malformed', "'junk' for delete_file: a token that does not decode gets the verifier's own reason: do not read its claims (not even its jti) before verify_token has passed"
    assert go('a.b', 'read_file') == 'denied:malformed', "'a.b' for read_file: a token that does not decode gets the verifier's own reason: do not read its claims (not even its jti) before verify_token has passed"
    assert go('a.b', 'delete_file') == 'denied:malformed', "'a.b' for delete_file: a token that does not decode gets the verifier's own reason: do not read its claims (not even its jti) before verify_token has passed"
    assert go('', 'read_file') == 'denied:malformed', "'' for read_file: a token that does not decode gets the verifier's own reason: do not read its claims (not even its jti) before verify_token has passed"
    assert go('', 'delete_file') == 'denied:malformed', "'' for delete_file: a token that does not decode gets the verifier's own reason: do not read its claims (not even its jti) before verify_token has passed"
    assert go(None, 'read_file') == 'denied:malformed', "None for read_file: a token that does not decode gets the verifier's own reason: do not read its claims (not even its jti) before verify_token has passed"
    assert go(None, 'delete_file') == 'denied:malformed', "None for delete_file: a token that does not decode gets the verifier's own reason: do not read its claims (not even its jti) before verify_token has passed"
    assert go('junk', 'read_events', srv=calendar) == 'denied:malformed', "'junk' for read_events: the verifier's own reason, with no claim read before verify_token has passed"
    assert go('a.b', 'read_events', srv=calendar) == 'denied:malformed', "'a.b' for read_events: the verifier's own reason, with no claim read before verify_token has passed"
    assert go('', 'read_events', srv=calendar) == 'denied:malformed', "'' for read_events: the verifier's own reason, with no claim read before verify_token has passed"
    assert go(None, 'read_events', srv=calendar) == 'denied:malformed', "None for read_events: the verifier's own reason, with no claim read before verify_token has passed"
    server = ResourceServer("files-api")
    size = len(server.audit)
    assert go(sign_token(make_claims(aud='mail-api')), 'read_file', now) == 'denied:wrong-audience', "expected Denied('wrong-audience'): the verifier's own reason comes through unchanged"
    assert go(sign_token(make_claims()), 'delete_file', now) == 'denied:missing-scope', "expected Denied('missing-scope'): the verifier's own reason comes through unchanged"
    assert go(sign_token(make_claims(exp=T0 + 5)), 'read_file', now) == 'denied:expired', "expected Denied('expired'): the verifier's own reason comes through unchanged"
    assert go(sign_token(make_claims(nbf=T0 + 50)), 'read_file', now) == 'denied:not-yet-valid', "expected Denied('not-yet-valid'): the verifier's own reason comes through unchanged"
    assert go(sign_token(make_claims(), SIGNING_KEY)[:-3] + 'xyz', 'read_file', now) == 'denied:bad-signature', "expected Denied('bad-signature'): the verifier's own reason comes through unchanged"
    assert go(sign_token(make_claims(aud="calendar-api", scope="calendar:readonly", jti="w-2")), "read_events", srv=calendar) == "denied:missing-scope", (
        "the scope is a whole-word test here too: calendar:readonly does not give calendar:read")
    assert go(sign_token(make_claims(scope='files:writeonly', jti='w-1')), 'delete_file') == 'denied:missing-scope', "files:writeonly does not give files:write (a whole-word test): delete_file is refused as 'missing-scope'"
    assert 'w-1' not in server.used, 'the call refused for files:writeonly spends nothing: the jti is not in server.used'
    assert len(server.audit) == size, "a refused call writes nothing to the audit"
    server = ResourceServer("files-api")
    server.revoked.add("x-3")
    assert go(sign_token(make_claims(jti="x-3")), "delete_file") == "denied:missing-scope", (
        "the verifier's scope test comes before revocation: a revoked token that also lacks the scope is 'missing-scope'")
    server.revoked.add("x-1")
    assert go(sign_token(make_claims(jti="x-1")), "read_file") == "denied:revoked", "a jti in self.revoked is 'revoked'"
    server.revoked.add("alice")
    assert go(sign_token(make_claims(jti="x-2")), "read_file") == "allowed", "revocation is by the token's jti only: a user name in the set revokes nothing"
    assert go(sign_token(make_claims(jti="x-1", exp=T0 + 5)), "read_file") == "denied:expired", "revocation is checked after the verifier, not before"
    shared = {"y-1"}
    s1, s2 = ResourceServer("files-api", SIGNING_KEY, shared), ResourceServer("files-api", SIGNING_KEY, shared)
    assert go(sign_token(make_claims(jti='y-1')), 'read_file', srv=s1) == 'denied:revoked', 'a jti added to the shared revocation set is revoked: look in self.revoked when the call is made'
    assert s1.revoked is s2.revoked, 'keep the revocation set you were given (self.revoked is that very set): do not copy it'
    server = ResourceServer("files-api")
    d1 = sign_token(make_claims(scope="files:write", jti="d-1"))
    assert go(d1, 'delete_file') == 'allowed', 'a first single-use call with an unspent token is allowed'
    assert set(server.used) == {'d-1'}, f'got {server.used}: a spent single-use token is recorded under its jti, and nothing else is recorded'
    assert server.used.get('d-1') == T0 + TTL, f'got {server.used}: a spent single-use token is recorded as jti -> exp'
    assert go(d1, "delete_file", now + 1) == "denied:replayed", "the same single-use token a second time is 'replayed'"
    used_before = dict(server.used)
    assert go(sign_token(make_claims(scope='files:write', jti='d-1', exp=T0 + 200)), 'delete_file', now + 1) == 'denied:replayed', "a token with the same jti and an exp of T0 + 200 is the same single-use token's jti: 'replayed' (the spent record is keyed on the jti alone)"
    assert server.used == used_before, f'got {server.used}, expected {used_before}: a refused call changes nothing in self.used, an exp of T0 + 200 included'
    used_before = dict(server.used)
    assert go(sign_token(make_claims(scope='files:write', jti='d-1', exp=T0 + 900)), 'delete_file', now + 1) == 'denied:replayed', "a token with the same jti and an exp of T0 + 900 is the same single-use token's jti: 'replayed' (the spent record is keyed on the jti alone)"
    assert server.used == used_before, f'got {server.used}, expected {used_before}: a refused call changes nothing in self.used, an exp of T0 + 900 included'
    assert go(sign_token(make_claims(scope="files:write", jti="d-2")), "delete_file") == "allowed", "a different jti is a different token"
    reader = sign_token(make_claims(jti="r-7"))
    assert go(reader, 'read_file') == 'allowed', 'a read is not single use: the same token may read again'
    assert go(reader, 'read_file') == 'allowed', 'a read is not single use: the same token may read again'
    assert go(reader, 'read_file') == 'allowed', 'a read is not single use: the same token may read again'
    assert go(reader, 'read_file') == 'allowed', 'a read is not single use: the same token may read again'
    assert "r-7" not in server.used, "only single-use operations record the token"
    fresh = ResourceServer("files-api")
    both = sign_token(make_claims(scope="files:read files:write", jti="b-1"))
    assert go(both, "read_file", srv=fresh) == "allowed", "a token that has not been used may read"
    assert go(both, "delete_file", srv=fresh) == "allowed", "reading with a token does not spend it for deleting"
    assert go(both, "delete_file", now + 1, fresh) == "denied:replayed", "deleting spends the token for deleting"
    assert go(both, "read_file", now + 2, fresh) == "allowed", (
        "a token spent on a single-use operation can still read: the replay test is for SINGLE_USE operations only")
    own = ResourceServer('calendar-api')
    token = sign_token(make_claims(aud='calendar-api', scope='calendar:read', jti='o-read_events'))
    assert go(token, 'read_events', srv=own) == 'allowed', 'read_events: the first call with a token that holds calendar:read on calendar-api is allowed'
    assert go(token, 'read_events', now + 1, own) == ('denied:replayed' if 'read_events' in SINGLE_USE else 'allowed'), 'read_events: ' + ("a single-use operation: the same token a second time is 'replayed'" if 'read_events' in SINGLE_USE else 'not a single-use operation: the same token may be used again (only the operations in SINGLE_USE are single use)')
    assert set(own.used) == ({'o-read_events'} if 'read_events' in SINGLE_USE else set()), f'got {own.used} after read_events twice: only the operations in SINGLE_USE record their token, under its jti'
    own = ResourceServer('calendar-api')
    token = sign_token(make_claims(aud='calendar-api', scope='calendar:write', jti='o-add_event'))
    assert go(token, 'add_event', srv=own) == 'allowed', 'add_event: the first call with a token that holds calendar:write on calendar-api is allowed'
    assert go(token, 'add_event', now + 1, own) == ('denied:replayed' if 'add_event' in SINGLE_USE else 'allowed'), 'add_event: ' + ("a single-use operation: the same token a second time is 'replayed'" if 'add_event' in SINGLE_USE else 'not a single-use operation: the same token may be used again (only the operations in SINGLE_USE are single use)')
    assert set(own.used) == ({'o-add_event'} if 'add_event' in SINGLE_USE else set()), f'got {own.used} after add_event twice: only the operations in SINGLE_USE record their token, under its jti'
    own = ResourceServer('files-api')
    token = sign_token(make_claims(aud='files-api', scope='files:read', jti='o-read_file'))
    assert go(token, 'read_file', srv=own) == 'allowed', 'read_file: the first call with a token that holds files:read on files-api is allowed'
    assert go(token, 'read_file', now + 1, own) == ('denied:replayed' if 'read_file' in SINGLE_USE else 'allowed'), 'read_file: ' + ("a single-use operation: the same token a second time is 'replayed'" if 'read_file' in SINGLE_USE else 'not a single-use operation: the same token may be used again (only the operations in SINGLE_USE are single use)')
    assert set(own.used) == ({'o-read_file'} if 'read_file' in SINGLE_USE else set()), f'got {own.used} after read_file twice: only the operations in SINGLE_USE record their token, under its jti'
    own = ResourceServer('files-api')
    token = sign_token(make_claims(aud='files-api', scope='files:write', jti='o-delete_file'))
    assert go(token, 'delete_file', srv=own) == 'allowed', 'delete_file: the first call with a token that holds files:write on files-api is allowed'
    assert go(token, 'delete_file', now + 1, own) == ('denied:replayed' if 'delete_file' in SINGLE_USE else 'allowed'), 'delete_file: ' + ("a single-use operation: the same token a second time is 'replayed'" if 'delete_file' in SINGLE_USE else 'not a single-use operation: the same token may be used again (only the operations in SINGLE_USE are single use)')
    assert set(own.used) == ({'o-delete_file'} if 'delete_file' in SINGLE_USE else set()), f'got {own.used} after delete_file twice: only the operations in SINGLE_USE record their token, under its jti'
    assert own.used.get('o-delete_file', T0 + TTL) == T0 + TTL, f'got {own.used} after delete_file twice: a recorded token is stored as jti -> exp'
    own = ResourceServer('mail-api')
    token = sign_token(make_claims(aud='mail-api', scope='mail:send', jti='o-send_mail'))
    assert go(token, 'send_mail', srv=own) == 'allowed', 'send_mail: the first call with a token that holds mail:send on mail-api is allowed'
    assert go(token, 'send_mail', now + 1, own) == ('denied:replayed' if 'send_mail' in SINGLE_USE else 'allowed'), 'send_mail: ' + ("a single-use operation: the same token a second time is 'replayed'" if 'send_mail' in SINGLE_USE else 'not a single-use operation: the same token may be used again (only the operations in SINGLE_USE are single use)')
    assert set(own.used) == ({'o-send_mail'} if 'send_mail' in SINGLE_USE else set()), f'got {own.used} after send_mail twice: only the operations in SINGLE_USE record their token, under its jti'
    assert own.used.get('o-send_mail', T0 + TTL) == T0 + TTL, f'got {own.used} after send_mail twice: a recorded token is stored as jti -> exp'
    mail = ResourceServer("mail-api")
    letter = sign_token(make_claims(aud="mail-api", scope="mail:send", jti="s-1"))
    assert go(letter, "send_mail", srv=mail) == "allowed", "a first send_mail is allowed"
    assert go(letter, "send_mail", now + 1, mail) == "denied:replayed", "every operation in SINGLE_USE is single use, send_mail too"
    assert len(mail.audit) == 1, "a replayed call is not audited"
    spent = ResourceServer("files-api")
    once = sign_token(make_claims(scope="files:write", jti="s-2"))
    assert go(once, "delete_file", srv=spent) == "allowed", "a first single-use delete_file with an unspent token is allowed"
    spent.revoked.add("s-2")
    assert go(once, "delete_file", now + 1, spent) == "denied:revoked", "revocation is tested before replay"
    gone = ResourceServer("files-api")
    wr = sign_token(make_claims(scope="files:write", jti="g-1"))
    gone.revoked.add("g-1")
    assert go(wr, 'delete_file', srv=gone) == 'denied:revoked', "a revoked token is refused as 'revoked'"
    assert gone.used == {}, 'a revoked call must not use up the token: self.used stays empty'
    gone.revoked.clear()
    assert go(wr, "delete_file", srv=gone) == "allowed", "the token was not spent by the refused call"
    other = ResourceServer("files-api")
    rd = sign_token(make_claims(scope="files:read", jti="m-1"))
    assert go(rd, 'delete_file', srv=other) == 'denied:missing-scope', "a token without the scope is refused as 'missing-scope'"
    assert other.used == {}, 'a call refused for scope must not use up the token: self.used stays empty'
    need = OPS['read_file'][1]
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-0')), 'nope', srv=sweep) == 'denied:unknown-op', "read_file, expected Denied('unknown-op')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'unknown-op' on read_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'unknown-op' on read_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-0')), 'send_mail', srv=sweep) == 'denied:unknown-op', "read_file, expected Denied('unknown-op')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'unknown-op' on read_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'unknown-op' on read_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go('junk', 'read_file', srv=sweep) == 'denied:malformed', "read_file, expected Denied('malformed')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'malformed' on read_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'malformed' on read_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-0'), alg='none'), 'read_file', srv=sweep) == 'denied:bad-alg', "read_file, expected Denied('bad-alg')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'bad-alg' on read_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'bad-alg' on read_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-0'), SIGNING_KEY)[:-3] + 'xyz', 'read_file', srv=sweep) == 'denied:bad-signature', "read_file, expected Denied('bad-signature')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'bad-signature' on read_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'bad-signature' on read_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-0', exp='x')), 'read_file', srv=sweep) == 'denied:bad-claims', "read_file, expected Denied('bad-claims')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'bad-claims' on read_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'bad-claims' on read_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-0', exp=T0 + 5)), 'read_file', srv=sweep) == 'denied:expired', "read_file, expected Denied('expired')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'expired' on read_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'expired' on read_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-0', nbf=T0 + 50)), 'read_file', srv=sweep) == 'denied:not-yet-valid', "read_file, expected Denied('not-yet-valid')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'not-yet-valid' on read_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'not-yet-valid' on read_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(aud='mail-api', scope=need, jti='z-0')), 'read_file', srv=sweep) == 'denied:wrong-audience', "read_file, expected Denied('wrong-audience')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'wrong-audience' on read_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'wrong-audience' on read_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope='calendar:read', jti='z-0')), 'read_file', srv=sweep) == 'denied:missing-scope', "read_file, expected Denied('missing-scope')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'missing-scope' on read_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'missing-scope' on read_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-1')), 'read_file', srv=sweep) == 'denied:revoked', "read_file, expected Denied('revoked')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'revoked' on read_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'revoked' on read_file it holds {sweep.used}"
    need = OPS['delete_file'][1]
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-0')), 'nope', srv=sweep) == 'denied:unknown-op', "delete_file, expected Denied('unknown-op')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'unknown-op' on delete_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'unknown-op' on delete_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-0')), 'send_mail', srv=sweep) == 'denied:unknown-op', "delete_file, expected Denied('unknown-op')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'unknown-op' on delete_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'unknown-op' on delete_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go('junk', 'delete_file', srv=sweep) == 'denied:malformed', "delete_file, expected Denied('malformed')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'malformed' on delete_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'malformed' on delete_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-0'), alg='none'), 'delete_file', srv=sweep) == 'denied:bad-alg', "delete_file, expected Denied('bad-alg')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'bad-alg' on delete_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'bad-alg' on delete_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-0'), SIGNING_KEY)[:-3] + 'xyz', 'delete_file', srv=sweep) == 'denied:bad-signature', "delete_file, expected Denied('bad-signature')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'bad-signature' on delete_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'bad-signature' on delete_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-0', exp='x')), 'delete_file', srv=sweep) == 'denied:bad-claims', "delete_file, expected Denied('bad-claims')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'bad-claims' on delete_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'bad-claims' on delete_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-0', exp=T0 + 5)), 'delete_file', srv=sweep) == 'denied:expired', "delete_file, expected Denied('expired')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'expired' on delete_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'expired' on delete_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-0', nbf=T0 + 50)), 'delete_file', srv=sweep) == 'denied:not-yet-valid', "delete_file, expected Denied('not-yet-valid')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'not-yet-valid' on delete_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'not-yet-valid' on delete_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(aud='mail-api', scope=need, jti='z-0')), 'delete_file', srv=sweep) == 'denied:wrong-audience', "delete_file, expected Denied('wrong-audience')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'wrong-audience' on delete_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'wrong-audience' on delete_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope='calendar:read', jti='z-0')), 'delete_file', srv=sweep) == 'denied:missing-scope', "delete_file, expected Denied('missing-scope')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'missing-scope' on delete_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'missing-scope' on delete_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-1')), 'delete_file', srv=sweep) == 'denied:revoked', "delete_file, expected Denied('revoked')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'revoked' on delete_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'revoked' on delete_file it holds {sweep.used}"
    sweep = ResourceServer('files-api')
    sweep.revoked.add('z-1')
    assert go(sign_token(make_claims(scope=need, jti='z-2')), 'delete_file', srv=sweep) == 'allowed', 'a first call with an unspent token is allowed'
    sweep.handle(sign_token(make_claims(scope='files:write', jti='z-9')), 'delete_file', now)
    audit_before, used_before = (list(sweep.audit), dict(sweep.used))
    assert go(sign_token(make_claims(scope=need, jti='z-2')), 'delete_file', srv=sweep) == 'denied:replayed', "delete_file, expected Denied('replayed')"
    assert sweep.audit == audit_before, f"a refused call, revoked included, writes nothing to the audit: after 'replayed' on delete_file it holds {sweep.audit}"
    assert sweep.used == used_before, f"a refused call spends nothing and records nothing in self.used: after 'replayed' on delete_file it holds {sweep.used}"
    prune = ResourceServer("files-api")
    long_lived = sign_token(make_claims(exp=T0 + 900, jti="p-0"))
    prune.handle(sign_token(make_claims(scope="files:write", jti="p-1")), "delete_file", T0 + 1)
    prune.handle(sign_token(make_claims(scope="files:write", jti="p-2", exp=T0 + 600)), "delete_file", T0 + 1)
    assert sorted(prune.used) == ["p-1", "p-2"], f"got {sorted(prune.used)}: two single-use deletes record two entries, keyed by jti (the clean-up drops only entries whose exp has passed)"
    prune.handle(long_lived, "read_file", T0 + TTL - 1)
    assert sorted(prune.used) == ["p-1", "p-2"], "an entry whose exp is still in the future stays"
    prune.handle(long_lived, "read_file", T0 + TTL)
    assert sorted(prune.used) == ["p-2"], f"got {sorted(prune.used)}: entries with exp <= now are dropped, so at now == exp the entry goes"
    assert _code(prune.handle, long_lived, 'nope', T0 + 600) == 'denied:unknown-op', "the clean-up is the first thing handle does, even for a call it then refuses: the call is still refused as 'unknown-op'"
    assert prune.used == {}, 'the clean-up is the first thing handle does, even for a call it then refuses: the entry whose exp has passed is gone'
    prune.handle(sign_token(make_claims(scope="files:write", jti="p-3", exp=T0 + 900)), "delete_file", T0 + 601)
    assert _code(prune.handle, 'junk', 'read_file', T0 + 900) == 'denied:malformed', "an undecodable token for a real operation still gets the verifier's own reason, 'malformed'"
    assert prune.used == {}, 'an undecodable token for a real operation still gets the clean-up first: the entry whose exp has passed is gone, and no claim was read before verify_token passed'
    print("exercise 3 looks right")


if __name__ == "__main__":
    _try("exercise 3", _check_server, needs=("exercise 1",))

## 6. Two designs, one list of attacks

Both designs sit behind the same five methods, so one list of attack scenarios can be run against either:

* `issue(user, agent, scopes, audience, now)`: the credential the agent is given (the delegated design may refuse and return `None`);
* `call(credential, audience, op, now)`: `True` when the service carried the operation out;
* `revoke(credential, now)`: the owner notices a leak and cancels that credential;
* `forge()`: what an attacker who knows no secret can write; `tamper(credential)`: a copy edited to ask for more.

**`StaticKeyDesign`** is the plain baseline many agents ship with: one random-looking key per user, kept in the agent's
configuration, accepted by every service for every operation, until someone deletes it. It ignores scopes, audiences and
lifetimes because it has none. (A key with its own scopes would do better; this is the *unscoped* key.)
**`TokenDesign`** is the one you built: login token, `exchange`, one `ResourceServer` per service, one shared revocation set.

A **scenario** is a function from a design to `True` when the attacker got what they wanted. Each is built from a
fresh design, so one attack cannot leave a revocation or a spent token behind for the next. The list was fixed before
any result was looked at; no attack was added or dropped afterwards.

In [ ]:
Scenario = namedtuple("Scenario", "name run")      # run(design) -> True when the attacker succeeded


class StaticKeyDesign:
    """One long-lived key per user, accepted by every service for every operation."""
    name = "static key"

    def __init__(self):
        self.owner = {"ak_" + hashlib.sha256(f"T07-L10:{user}".encode()).hexdigest()[:24]: user for user in USERS}

    def issue(self, user, agent, scopes, audience, now):
        return next(key for key, who in self.owner.items() if who == user)      # the user's own key, whatever was asked

    def call(self, credential, audience, op, now):
        return credential in self.owner and op in OPS and OPS[op][0] == audience

    def revoke(self, credential, now):
        self.owner.pop(credential, None)                                         # rotate: that key stops working for everyone

    def forge(self):
        return "ak_" + "0" * 24

    def tamper(self, credential):
        return credential[:-1] + ("0" if credential[-1] != "0" else "1")


class TokenDesign:
    """Login token, exchange, one resource server per service, one shared revocation set."""
    name = "delegated tokens"

    def __init__(self):
        self.revoked = set()
        self.serial = 0
        self.servers = {audience: ResourceServer(audience, SIGNING_KEY, self.revoked) for audience in AUDIENCES}

    def _jti(self, who):
        self.serial += 1
        return f"{who}-{self.serial:04d}"

    def issue(self, user, agent, scopes, audience, now):
        login = login_token(user, iat=now, exp=now + SUBJECT_TTL, jti=self._jti("login-" + user))
        try:
            return exchange(login, agent, scopes, audience, now, self._jti(agent))
        except Denied:
            return None

    def call(self, credential, audience, op, now):
        if credential is None or audience not in self.servers:
            return False
        try:
            self.servers[audience].handle(credential, op, now)
        except Denied:
            return False
        return True

    def revoke(self, credential, now):
        self.revoked.add(peek_claims(credential)["jti"])

    def forge(self):
        claims = {"sub": "alice", "aud": "files-api", "scope": "files:read files:write", "exp": T0 + 10 * DAY, "jti": "forged-1"}
        return f"{_b64(_json({'alg': 'none', 'typ': 'JWT'}))}.{_b64(_json(claims))}."

    def tamper(self, credential):
        head, _, sig = credential.split(".")
        claims = peek_claims(credential)
        claims["scope"] += " files:write"
        return f"{head}.{_b64(_json(claims))}.{sig}"


def _agent(design, scopes, audience, user="alice", agent="agent-scribe"):
    return design.issue(user, agent, scopes, audience, T0)


def forged_credential(d):
    """The attacker writes a credential from nothing and tries to delete a file."""
    return d.call(d.forge(), "files-api", "delete_file", T0 + 5)


def tampered_credential(d):
    """The attacker edits a read-only credential to ask for more, then deletes a file."""
    return d.call(d.tamper(_agent(d, ["files:read"], "files-api")), "files-api", "delete_file", T0 + 5)


def stale_leak(d):
    """A credential leaked into a log is used thirty days later."""
    return d.call(_agent(d, ["files:read"], "files-api"), "files-api", "read_file", T0 + 30 * DAY)


def fresh_leak(d):
    """The same leak, used one minute after it was issued."""
    return d.call(_agent(d, ["files:read"], "files-api"), "files-api", "read_file", T0 + 60)


def leak_then_revoke(d):
    """A leaked credential; the owner revokes it after ten seconds; the attacker uses it at one minute."""
    cred = _agent(d, ["files:read"], "files-api")
    d.revoke(cred, T0 + 10)
    return d.call(cred, "files-api", "read_file", T0 + 60)


def wrong_service(d):
    """A calendar credential is presented to the mail service to send mail."""
    return d.call(_agent(d, ["calendar:read"], "calendar-api"), "mail-api", "send_mail", T0 + 5)


def scope_escalation(d):
    """A read-only files credential is used to delete a file."""
    return d.call(_agent(d, ["files:read"], "files-api"), "files-api", "delete_file", T0 + 5)


def exchange_escalation(d):
    """The agent asks for calendar write access, which its registration does not allow, then adds an event."""
    cred = _agent(d, ["calendar:write"], "calendar-api")
    return cred is not None and d.call(cred, "calendar-api", "add_event", T0 + 5)


def rogue_agent(d):
    """A program nobody registered asks for a files credential on alice's behalf, then deletes a file."""
    cred = _agent(d, ["files:write"], "files-api", agent="agent-rogue")
    return cred is not None and d.call(cred, "files-api", "delete_file", T0 + 5)


def bob_overreach(d):
    """The agent acting for bob, who may only read the calendar, asks for file deletion and deletes a file."""
    cred = _agent(d, ["files:write"], "files-api", user="bob")
    return cred is not None and d.call(cred, "files-api", "delete_file", T0 + 5)


def replayed_delete(d):
    """A captured delete request is sent again after the real one went through."""
    cred = _agent(d, ["files:write"], "files-api")
    d.call(cred, "files-api", "delete_file", T0 + 5)
    return d.call(cred, "files-api", "delete_file", T0 + 6)


def replayed_read(d):
    """A captured read request is sent again after the real one went through."""
    cred = _agent(d, ["files:read"], "files-api")
    d.call(cred, "files-api", "read_file", T0 + 5)
    return d.call(cred, "files-api", "read_file", T0 + 6)


SCENARIOS = [Scenario(f.__name__, f) for f in (
    forged_credential, tampered_credential, stale_leak, fresh_leak, leak_then_revoke, wrong_service,
    scope_escalation, exchange_escalation, rogue_agent, bob_overreach, replayed_delete, replayed_read)]


def show_scenarios() -> None:
    print(f"{len(SCENARIOS)} scenarios, each run on a fresh design:")
    for s in SCENARIOS:
        print(f"  {s.name:22} {s.run.__doc__}")


if __name__ == "__main__":
    show_scenarios()

## 7. Exercise 4: run the attacks and count

**`run_suite(make_design, scenarios)`** returns a dict from each scenario's `name` to `bool(scenario.run(design))`
(`True`: the attacker succeeded), in the order of `scenarios`. It calls `make_design()` **once per scenario**, so every
scenario gets a fresh design and nothing one attack changes is visible to the next.

**`compare(static_results, token_results)`** takes two such dicts and returns a dict of four lists of scenario names,
each in the order of `static_results`, under four keys that come in exactly this order: `"both-stop"` (the attacker failed against both designs), `"only-token-stops"`
(the attacker beat the static key and failed against the tokens), `"only-static-stops"` (the reverse) and
`"neither-stops"` (the attacker succeeded against both). It raises `ValueError` when the two dicts do not hold exactly
the same names (the same names in a different order are fine). Never change the dicts. The four keys of the answer come
in the order listed above (`both-stop` first, `neither-stops` last), so that two answers print alike.

<details><summary>💡 Hint 1 — what to think about</summary>

What would a revocation made in one scenario do to the next one if both ran on the same design? A value of `True`
means the attacker *won*, so a design stops an attack when the value is `False`: keep that straight when you sort
the names into the four lists.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

A dict comprehension over the scenarios, calling the factory inside it. For `compare`, check the two key sets first,
then loop over the static results in order and decide the list from the two booleans.

</details>

In [ ]:
def run_suite(make_design: Callable, scenarios: list) -> dict:
    """The outcome of every scenario on its own fresh design, as described above.

    Example:
        >>> run_suite(StaticKeyDesign, [Scenario("a", lambda d: True), Scenario("b", lambda d: 0)])
        {'a': True, 'b': False}

    Returns:
        A dict from scenario name to True (the attacker succeeded) or False.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def compare(static_results: dict, token_results: dict) -> dict:
    """Sort the scenario names by which design stopped them, as described above.

    Example:
        >>> compare({"x": True, "y": False}, {"x": False, "y": False})
        {'both-stop': ['y'], 'only-token-stops': ['x'], 'only-static-stops': [], 'neither-stops': []}

    Returns:
        A dict with the four keys, in the order described above, each a list of names.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_suite() -> None:
    made = []

    def factory():
        made.append(StaticKeyDesign())
        return made[-1]

    seen = []

    def script(answer, mark=None):
        def run(design):
            seen.append(design)
            if mark:
                mark(design)
            return answer
        return run

    class OneShot:
        """A stand-in design (it uses none of your code): a single-use call spends the credential, and a revocation sticks."""
        name = "one-shot stand-in"

        def __init__(self):
            self.spent, self.revoked = set(), set()

        def issue(self, user, agent, scopes, audience, now):
            return f"cred:{user}:{agent}"

        def call(self, credential, audience, op, now):
            if credential in self.revoked or credential in self.spent:
                return False
            if op in SINGLE_USE:
                self.spent.add(credential)
            return True

        def revoke(self, credential, now):
            self.revoked.add(credential)

        def forge(self):
            return "cred:forged"

        def tamper(self, credential):
            return credential + ":edited"

    def spend(design):
        cred = design.issue("alice", "agent-scribe", ["files:write"], "files-api", T0)
        design.call(cred, "files-api", "delete_file", T0 + 1)
        design.revoke(cred, T0 + 2)
        return False
    def again(design):
        return design.call(design.issue("alice", "agent-scribe", ["files:write"], "files-api", T0), "files-api", "delete_file", T0 + 3)
    sa = {"p": True, "q": False, "r": True, "s": False, "t": True}
    tk = {"p": False, "q": False, "r": True, "s": True, "t": False}
    want = {"both-stop": ["q"], "only-token-stops": ["p", "t"], "only-static-stops": ["s"], "neither-stops": ["r"]}
    keep_a, keep_b = dict(sa), dict(tk)
    fates = {"z": (False, False), "t": (True, False), "q": (False, True), "m": (True, True), "c": (False, False), "k": (True, False), "b": (False, True), "h": (True, True)}
    got = run_suite(factory, [Scenario("one", script(True)), Scenario("two", script(False)), Scenario("three", script(5)), Scenario("four", script([]))])
    assert got["one"] == True, f"got {got}: bool(run(design)) for a scenario that returns True"
    assert got["two"] == False, f"got {got}: bool(run(design)) for a scenario that returns False"
    assert got["three"] == True, f"got {got}: bool(run(design)) for a scenario that returns 5"
    assert got["four"] == False, f"got {got}: bool(run(design)) for a scenario that returns an empty list"
    assert type(got["one"]) is bool, "the value for one is bool(run(design)), a bool"
    assert type(got["two"]) is bool, "the value for two is bool(run(design)), a bool"
    assert type(got["three"]) is bool, "the value for three is bool(run(design)), a bool, not the 5 the scenario returned"
    assert type(got["four"]) is bool, "the value for four is bool(run(design)), a bool, not the empty list the scenario returned"
    assert list(got) == ["one", "two", "three", "four"], "keep the order of the scenarios"
    assert len(made) == 4, 'call make_design() once for each scenario (four scenarios, four designs)'
    assert [id(d) for d in seen] == [id(d) for d in made], 'give each scenario its own design, in order: the designs the scenarios saw are the designs make_design() made, in that order'
    del made[:]
    empty = run_suite(factory, [])
    assert empty == {}, "no scenarios: no results"
    assert made == [], "no scenarios: make_design() is never called (one design for each scenario, none for none)"
    del made[:]
    cut = Scenario("cut", script(False, lambda d: d.revoke(d.issue("alice", "a", [], "files-api", T0), T0)))
    check = Scenario("check", lambda d: d.call(d.issue("alice", "a", [], "files-api", T0), "files-api", "read_file", T0))
    cut_check = run_suite(StaticKeyDesign, [cut, check])
    assert cut_check["cut"] == False, "the first scenario returns False whatever it revoked: bool(run(design))"
    assert cut_check["check"] == True, "the second scenario must not see the first one's revocation: a fresh design for each"
    assert list(cut_check) == ["cut", "check"], "one entry per scenario, in the order of the scenarios"




    spend_again_static = run_suite(StaticKeyDesign, [Scenario('spend', spend), Scenario('again', again)])
    assert spend_again_static['spend'] == False, f'{StaticKeyDesign.__name__}: the first scenario returns False whatever it spent or revoked: bool(run(design))'
    assert spend_again_static['again'] == True, f'{StaticKeyDesign.__name__}: a spent credential or a revocation from the first scenario is visible to the second: call make_design() again for every scenario'
    assert list(spend_again_static) == ['spend', 'again'], f'{StaticKeyDesign.__name__}: one entry per scenario, in the order of the scenarios'
    spend_again_one_shot = run_suite(OneShot, [Scenario('spend', spend), Scenario('again', again)])
    assert spend_again_one_shot['spend'] == False, f'{OneShot.__name__}: the first scenario returns False whatever it spent or revoked: bool(run(design))'
    assert spend_again_one_shot['again'] == True, f'{OneShot.__name__}: a spent credential or a revocation from the first scenario is visible to the second: call make_design() again for every scenario'
    assert list(spend_again_one_shot) == ['spend', 'again'], f'{OneShot.__name__}: one entry per scenario, in the order of the scenarios'
    got = compare(dict(sa), dict(tk))
    assert got["both-stop"] == want["both-stop"], f"got {got}: attacker True means the design FAILED; both-stop holds the names whose attacker failed against both designs"
    assert got["only-token-stops"] == want["only-token-stops"], f"got {got}: attacker True means the design FAILED; static fails and token holds is only-token-stops"
    assert got["only-static-stops"] == want["only-static-stops"], f"got {got}: attacker True means the design FAILED; static holds and token fails is only-static-stops"
    assert got["neither-stops"] == want["neither-stops"], f"got {got}: attacker True means the design FAILED; neither-stops holds the names that beat both designs"
    assert list(got) == list(want), f"the lists are right but the keys are in the order {list(got)}: the four keys come in exactly the order both-stop, only-token-stops, only-static-stops, neither-stops"
    assert compare(dict(reversed(list(sa.items()))), dict(tk))["only-token-stops"] == ["t", "p"], "names come out in the order of static_results"
    nothing = compare({}, {})
    assert nothing["both-stop"] == [], "empty in, four empty lists out: both-stop"
    assert nothing["only-token-stops"] == [], "empty in, four empty lists out: only-token-stops"
    assert nothing["only-static-stops"] == [], "empty in, four empty lists out: only-static-stops"
    assert nothing["neither-stops"] == [], "empty in, four empty lists out: neither-stops"
    assert len(nothing) == 4, "empty in, exactly four lists out: both-stop, only-token-stops, only-static-stops and neither-stops, and no other key"
    got = compare({n: f[0] for n, f in fates.items()}, {n: f[1] for n, f in fates.items()})
    assert got["both-stop"] == ["z", "c"], f"got {got}: in every one of the four lists the names keep the order they have in static_results (none of the lists is sorted, reversed or built in another order): both-stop"
    assert got["only-token-stops"] == ["t", "k"], f"got {got}: in every one of the four lists the names keep the order they have in static_results (none of the lists is sorted, reversed or built in another order): only-token-stops"
    assert got["only-static-stops"] == ["q", "b"], f"got {got}: in every one of the four lists the names keep the order they have in static_results (none of the lists is sorted, reversed or built in another order): only-static-stops"
    assert got["neither-stops"] == ["m", "h"], f"got {got}: in every one of the four lists the names keep the order they have in static_results (none of the lists is sorted, reversed or built in another order): neither-stops"
    compare(sa, tk)
    assert sa == keep_a, "never change the first dict (static_results)"
    assert tk == keep_b, "never change the second dict (token_results)"
    same = compare(dict(keep_a), dict(reversed(list(keep_b.items()))))
    assert same["both-stop"] == want["both-stop"], "the same names listed in a different order are the same scenarios: no ValueError, and the lists keep the order of static_results: both-stop"
    assert same["only-token-stops"] == want["only-token-stops"], "the same names listed in a different order are the same scenarios: no ValueError, and the lists keep the order of static_results: only-token-stops"
    assert same["only-static-stops"] == want["only-static-stops"], "the same names listed in a different order are the same scenarios: no ValueError, and the lists keep the order of static_results: only-static-stops"
    assert same["neither-stops"] == want["neither-stops"], "the same names listed in a different order are the same scenarios: no ValueError, and the lists keep the order of static_results: neither-stops"
    assert _error_of(compare, {'a': True}, {'b': True}) == 'ValueError', "{'a': True} versus {'b': True}: results that do not cover the same scenarios must raise ValueError"
    assert _error_of(compare, {'a': True}, {'a': True, 'b': False}) == 'ValueError', "{'a': True} versus {'a': True, 'b': False}: results that do not cover the same scenarios must raise ValueError"
    assert _error_of(compare, {'a': True, 'b': False}, {'a': True}) == 'ValueError', "{'a': True, 'b': False} versus {'a': True}: results that do not cover the same scenarios must raise ValueError"
    assert _error_of(compare, {'a': True, 'b': False}, {'b': False}) == 'ValueError', "{'a': True, 'b': False} versus {'b': False}: results that do not cover the same scenarios must raise ValueError"
    assert _error_of(compare, {}, {'a': True}) == 'ValueError', "{} versus {'a': True}: results that do not cover the same scenarios must raise ValueError"
    want_static = {s.name: bool(s.run(StaticKeyDesign())) for s in SCENARIOS}
    want_other = {s.name: bool(s.run(OneShot())) for s in SCENARIOS}
    names = [s.name for s in SCENARIOS]
    static = run_suite(StaticKeyDesign, SCENARIOS)
    assert static['forged_credential'] == want_static['forged_credential'], "forged_credential on a fresh StaticKeyDesign: the same result as running the scenario on its own new design"
    assert static['tampered_credential'] == want_static['tampered_credential'], "tampered_credential on a fresh StaticKeyDesign: the same result as running the scenario on its own new design"
    assert static['stale_leak'] == want_static['stale_leak'], "stale_leak on a fresh StaticKeyDesign: the same result as running the scenario on its own new design"
    assert static['fresh_leak'] == want_static['fresh_leak'], "fresh_leak on a fresh StaticKeyDesign: the same result as running the scenario on its own new design"
    assert static['leak_then_revoke'] == want_static['leak_then_revoke'], "leak_then_revoke on a fresh StaticKeyDesign: the same result as running the scenario on its own new design"
    assert static['wrong_service'] == want_static['wrong_service'], "wrong_service on a fresh StaticKeyDesign: the same result as running the scenario on its own new design"
    assert static['scope_escalation'] == want_static['scope_escalation'], "scope_escalation on a fresh StaticKeyDesign: the same result as running the scenario on its own new design"
    assert static['exchange_escalation'] == want_static['exchange_escalation'], "exchange_escalation on a fresh StaticKeyDesign: the same result as running the scenario on its own new design"
    assert static['rogue_agent'] == want_static['rogue_agent'], "rogue_agent on a fresh StaticKeyDesign: the same result as running the scenario on its own new design"
    assert static['bob_overreach'] == want_static['bob_overreach'], "bob_overreach on a fresh StaticKeyDesign: the same result as running the scenario on its own new design"
    assert static['replayed_delete'] == want_static['replayed_delete'], "replayed_delete on a fresh StaticKeyDesign: the same result as running the scenario on its own new design"
    assert static['replayed_read'] == want_static['replayed_read'], "replayed_read on a fresh StaticKeyDesign: the same result as running the scenario on its own new design"
    assert list(static) == names, "one entry per scenario, in the order of the scenarios (static key design)"
    token = run_suite(OneShot, SCENARIOS)
    assert token['forged_credential'] == want_other['forged_credential'], "forged_credential on a fresh one-shot design: the same result as running the scenario on its own new design"
    assert token['tampered_credential'] == want_other['tampered_credential'], "tampered_credential on a fresh one-shot design: the same result as running the scenario on its own new design"
    assert token['stale_leak'] == want_other['stale_leak'], "stale_leak on a fresh one-shot design: the same result as running the scenario on its own new design"
    assert token['fresh_leak'] == want_other['fresh_leak'], "fresh_leak on a fresh one-shot design: the same result as running the scenario on its own new design"
    assert token['leak_then_revoke'] == want_other['leak_then_revoke'], "leak_then_revoke on a fresh one-shot design: the same result as running the scenario on its own new design"
    assert token['wrong_service'] == want_other['wrong_service'], "wrong_service on a fresh one-shot design: the same result as running the scenario on its own new design"
    assert token['scope_escalation'] == want_other['scope_escalation'], "scope_escalation on a fresh one-shot design: the same result as running the scenario on its own new design"
    assert token['exchange_escalation'] == want_other['exchange_escalation'], "exchange_escalation on a fresh one-shot design: the same result as running the scenario on its own new design"
    assert token['rogue_agent'] == want_other['rogue_agent'], "rogue_agent on a fresh one-shot design: the same result as running the scenario on its own new design"
    assert token['bob_overreach'] == want_other['bob_overreach'], "bob_overreach on a fresh one-shot design: the same result as running the scenario on its own new design"
    assert token['replayed_delete'] == want_other['replayed_delete'], "replayed_delete on a fresh one-shot design: the same result as running the scenario on its own new design"
    assert token['replayed_read'] == want_other['replayed_read'], "replayed_read on a fresh one-shot design: the same result as running the scenario on its own new design"
    assert list(token) == names, "one entry per scenario, in the order of the scenarios (one-shot design)"
    lists = {(False, False): "both-stop", (True, False): "only-token-stops", (False, True): "only-static-stops", (True, True): "neither-stops"}
    groups = compare(want_static, want_other)
    assert sum(len(v) for v in groups.values()) == len(SCENARIOS), "every scenario lands in one list: the four lists together hold each name once"
    assert [key for key, listed in groups.items() if 'forged_credential' in listed] == [lists[(want_static['forged_credential'], want_other['forged_credential'])]], "forged_credential must be in exactly one list, the one that says which designs the attacker beat"
    assert [key for key, listed in groups.items() if 'tampered_credential' in listed] == [lists[(want_static['tampered_credential'], want_other['tampered_credential'])]], "tampered_credential must be in exactly one list, the one that says which designs the attacker beat"
    assert [key for key, listed in groups.items() if 'stale_leak' in listed] == [lists[(want_static['stale_leak'], want_other['stale_leak'])]], "stale_leak must be in exactly one list, the one that says which designs the attacker beat"
    assert [key for key, listed in groups.items() if 'fresh_leak' in listed] == [lists[(want_static['fresh_leak'], want_other['fresh_leak'])]], "fresh_leak must be in exactly one list, the one that says which designs the attacker beat"
    assert [key for key, listed in groups.items() if 'leak_then_revoke' in listed] == [lists[(want_static['leak_then_revoke'], want_other['leak_then_revoke'])]], "leak_then_revoke must be in exactly one list, the one that says which designs the attacker beat"
    assert [key for key, listed in groups.items() if 'wrong_service' in listed] == [lists[(want_static['wrong_service'], want_other['wrong_service'])]], "wrong_service must be in exactly one list, the one that says which designs the attacker beat"
    assert [key for key, listed in groups.items() if 'scope_escalation' in listed] == [lists[(want_static['scope_escalation'], want_other['scope_escalation'])]], "scope_escalation must be in exactly one list, the one that says which designs the attacker beat"
    assert [key for key, listed in groups.items() if 'exchange_escalation' in listed] == [lists[(want_static['exchange_escalation'], want_other['exchange_escalation'])]], "exchange_escalation must be in exactly one list, the one that says which designs the attacker beat"
    assert [key for key, listed in groups.items() if 'rogue_agent' in listed] == [lists[(want_static['rogue_agent'], want_other['rogue_agent'])]], "rogue_agent must be in exactly one list, the one that says which designs the attacker beat"
    assert [key for key, listed in groups.items() if 'bob_overreach' in listed] == [lists[(want_static['bob_overreach'], want_other['bob_overreach'])]], "bob_overreach must be in exactly one list, the one that says which designs the attacker beat"
    assert [key for key, listed in groups.items() if 'replayed_delete' in listed] == [lists[(want_static['replayed_delete'], want_other['replayed_delete'])]], "replayed_delete must be in exactly one list, the one that says which designs the attacker beat"
    assert [key for key, listed in groups.items() if 'replayed_read' in listed] == [lists[(want_static['replayed_read'], want_other['replayed_read'])]], "replayed_read must be in exactly one list, the one that says which designs the attacker beat"
    print("exercise 4 looks right")


if __name__ == "__main__":
    _try("exercise 4", _check_suite)

## 8. The measurement

Now run every scenario against both designs and let the code say what it counted. The table is exact: the same
scenarios, the same simulated clock, no randomness. Below it, the reason each delegated-token refusal gave, taken
from the servers rather than typed, and the audit line the files service wrote.

In [ ]:
def _reason(call, *args):
    try:
        call(*args)
    except Denied as exc:
        return exc.code
    return "allowed"


def _measure() -> None:
    static = run_suite(StaticKeyDesign, SCENARIOS)
    token = run_suite(TokenDesign, SCENARIOS)
    print(f"{'scenario':22} {'static key':>14} {'delegated tokens':>18}")
    for s in SCENARIOS:
        print(f"{s.name:22} {'attacker wins' if static[s.name] else 'stopped':>14} {'attacker wins' if token[s.name] else 'stopped':>18}")
    groups = compare(static, token)
    total = len(SCENARIOS)
    print(f"\nstatic key stops {sum(not v for v in static.values())} of {total}; delegated tokens stop {sum(not v for v in token.values())} of {total}")
    for key, names in groups.items():
        print(f"  {key}: {len(names)}  {', '.join(names) if names else '-'}")
    d = TokenDesign()
    tok = _agent(d, ["files:read"], "files-api")
    probes = [("used 30 days later", d.servers["files-api"].handle, tok, "read_file", T0 + 30 * DAY),
              ("sent to the mail service", d.servers["mail-api"].handle, tok, "send_mail", T0 + 5),
              ("used to delete a file", d.servers["files-api"].handle, tok, "delete_file", T0 + 5)]
    print("\nwhy a delegated token was refused:")
    for label, handle, *args in probes:
        print(f"  {label:26} -> {_reason(handle, *args)}")
    login = login_token()
    print(f"  {'asked for calendar:write':26} -> {_reason(exchange, login, 'agent-scribe', ['calendar:write'], 'calendar-api', T0, 'j')}")
    d.servers["files-api"].handle(tok, "read_file", T0 + 5)
    print("\nthe files service's audit of the call that was allowed:", d.servers["files-api"].audit)
    static = StaticKeyDesign()
    key = static.issue("alice", "agent-scribe", ["files:read"], "files-api", T0)
    print(f"the static key's gateway can attribute that call to {static.owner[key]!r} only; its state is {sorted(vars(static))}, "
          f"and a field that names an agent: {any('agent' in name for name in vars(static))}")


if __name__ == "__main__":
    _try("measurement", _measure, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 9. What this measurement tells you, and what it does not

* **It counts attacks, not safety.** The list is the author's, fixed in advance. A different list gives different counts;
  the counts describe these two designs against these attacks.
* **The static key is the plain baseline.** A key limited to one service, with its own scopes and a rotation schedule,
  would stop more. The comparison shows what the *unscoped* long-lived key costs; a real configuration may do better
  or worse, and this lesson does not count how many do either.
* **Short lifetimes bound a theft; they do not prevent its use.** A bearer token works for whoever holds it until it
  expires or is revoked. Read the neither-stops line printed above and check what its scenarios have in common: each uses a
  credential inside its lifetime, and no check built here can tell the thief from the agent.
* **Single use was applied to destructive operations only.** Replaying a read is allowed here because blocking every
  repeat read would break normal agents; a stricter service may choose otherwise.
* **One shared HMAC key is a teaching shortcut.** Every service that can verify a token could also mint one. A real
  deployment uses asymmetric signatures (services hold only a public key) or asks the token service to introspect
  each token, runs a maintained library rather than this code, and keeps keys in a secrets manager.
* **The MCP specification is moving.** Section 13 quotes the revision current when this lesson was written, with the
  date it was read. This lesson builds the mechanics that revision asks of a server (audience validation, no accepting
  tokens issued for another service); it does not test an MCP server.

## 10. Common mistakes

* **Reading claims before verifying the signature.** The signature check is first for a reason: every claim is the sender's words until it passes.
* **Obeying the header's `alg`.** A token that says its algorithm is `none` asks you to skip the check; the verifier chooses the algorithm, never the token.
* **Comparing signatures with `==`.** Use a constant-time comparison; and compare text you were sent as the bytes you computed.
* **Substring scope tests.** `files:readonly` contains `files:read`; scopes are whole words.
* **Trimming an escalation instead of refusing it.** An agent that asked for more than it may have should hear no, once, from the exchange.
* **Spending a token on a refused call.** A call refused for scope, audience or revocation must leave the token usable for what it was allowed to do.
* **Letting the replay cache grow for ever.** Entries past their `exp` can never matter again.

## 11. What this lesson does not cover

* **A real authorization server.** Login, consent screens, refresh tokens, key rotation and discovery are not built.
* **Asymmetric keys, token binding to a sender, or introspection.** Each narrows the stolen-token gap this lesson measures; none is built.
* **Agents calling agents.** A chain of delegations, nested `act` claims, and the policy for them are not built.

In [ ]:
# Facts about the platform you just used, printed by the code that runs here, for the self-check below.
if __name__ == "__main__":
    print(f"an agent token lives at most {TTL} s; a login token lives {SUBJECT_TTL} s; {len(SINGLE_USE)} of the {len(OPS)} operations are single use")
    print(f"the agent may hold {len(AGENTS['agent-scribe']['scopes'])} scopes; alice's login holds {len(USERS['alice'])} and bob's holds {len(USERS['bob'])}")
    print(f"the platform has {len(USERS)} users, {len(AGENTS)} registered agent, {len(AUDIENCES)} services and {len(OPS)} operations")

## 12. Self-check

1. A delegated token is stolen shortly after it was issued and is used again while it is still inside its short lifetime. Which of these can stop the thief?
   - (a) Its expiry, because every token expires before it can be misused.
   - (b) Its audience claim, because the thief is a different person from the agent.
   - (c) A revocation the owner has already made, which the service checks on every call.
   - (d) Its `act` claim, because a token that names an agent cannot be used by anyone else.

2. What does the `act` claim add to a token that already has `sub`?
   - (a) Authority: the agent may do what the agent's own registration allows, on top of the user's scopes.
   - (b) A record, for the service and its audit log, of who is exercising the user's authority; the authority itself is still limited by `scope`.
   - (c) Protection against forgery, since the actor's name is covered by a second signature.
   - (d) A replacement for `aud`, because the actor now identifies the service that should accept the token.

3. Why does the resource server drop spent-token entries whose `exp` has passed?
   - (a) Because an expired token could be replayed once more if its entry were kept.
   - (b) Because spent tokens must be forgotten after one use so they can be used again later.
   - (c) Because the clock must be checked before the signature, and the cache is the clock.
   - (d) Because a replay of an expired token is refused as expired anyway, so the entry only occupies memory.

4. Suppose a measurement like this one showed both designs losing a scenario in which the credential is used inside its lifetime. What is the most defensible reading?
   - (a) A credential works for whoever holds it while it is valid and unrevoked; lifetimes shorter than the ones used here, and binding it to its sender, narrow that; this suite tested neither.
   - (b) Tokens are no better than static keys, so the extra checks are not worth building.
   - (c) HMAC is too weak a signature, and the failure would disappear with a longer key.
   - (d) The scenarios are wrongly written, because a good design stops every attack.

Answers, with reasoning, come with this lesson's worked solution when you enrol on Synapsa.

## 13. Sources, with the date each was read
Every dated statement above is a `claims.yaml` entry with its source, a verbatim quote and the day it was read
(13 November 2026). Specifications and standards change: read the pages again before you rely on them.
* Model Context Protocol, specification revision 2026-07-28 (the latest on the site that day): authorization (audience validation, no accepting other tokens) and security best practices (token passthrough, scope minimisation).
* IETF RFC 8693, OAuth 2.0 Token Exchange: the `act` claim.
* IETF RFC 7519, JSON Web Token: `exp`, `aud` and `jti`, and what an unsecured token (the `none` algorithm) is.
* IETF RFC 8725, JSON Web Token Best Current Practices: the caller, not the token's header, fixes the accepted algorithms.

In [ ]:
# Your progress board: one line per exercise, from the checks you have run, then the count.
_MARKS = {"passed": "✅ passed     ", "failed": "❌ failed     ", "not started": "⏳ not started"}


def _progress_board() -> None:
    """One line per exercise, from what the checks recorded, then how many are complete."""
    print("\nyour progress")
    done, next_up = 0, None
    for label, shown, stubs, where in _EXERCISES:
        status = _STATUS.get(label, "not started")
        done += status == "passed"
        waiting = _WAITING_ON.get(label)
        note = (f"   (waiting on `{waiting}`)" if status == "not started" and waiting not in (None, "") and waiting not in stubs
                else "")
        print(f"  {_MARKS[status]}  {label} · {shown}{note}")
        if status != "passed" and next_up is None:
            next_up = (label, shown, where)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    if next_up:
        print(f"next: {next_up[0]}, {next_up[1]}, in {next_up[2]}. Its cell has hints if you are stuck.")


def _finish() -> None:
    """The board, then the verdict. A stub you have not reached yet is not a failure; a check that ran and
    came back wrong is. In a script or under CI that ends the run non-zero; in a notebook kernel it is a printed line."""
    _progress_board()
    failed = [label for label, status in _STATUS.items() if status == "failed"]
    if failed and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(failed))
    if failed:
        print("checks failed: " + ", ".join(failed) + " — each one printed its likely mistake where it ran.")


if __name__ == "__main__":
    _finish()

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.